# 27 Perbandingan Pembentukan Profil


In [1]:
from pathlib import Path
import os, types
from IPython.display import display

PROJECT_ROOT = '/content/drive/MyDrive/REVISED'
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', force_remount=False)
ROOT_27 = Path(os.environ.get('PROFILE27_ROOT', PROJECT_ROOT)).expanduser().resolve()
if not (ROOT_27 / 'LIB').is_dir():
    raise FileNotFoundError('Folder proyek belum tersedia: ' + str(ROOT_27))
print('Proyek:', ROOT_27)


Mounted at /content/drive
Proyek: /content/drive/MyDrive/REVISED


## Operator yang dibandingkan

M adalah balance evidence mode; B adalah balance evidence anggota (P−N).
Kamus evidence, perlakuan netral, threshold, fitur, split, dan partisi tetap.

| Operator | Arah profil | Ketika seri |
|---|---|---|
| PRIMARY_MODE_FIRST | tanda M; bila nol, tanda B | keduanya nol → unresolved |
| DIRECT_BEHAVIORAL_RULES | jumlah signed evidence indikator peserta itu sendiri, tanpa cluster | nol → unresolved |
| MEMBER_ONLY | tanda B, tanpa prioritas mode | B nol → unresolved |
| DIRECTION_CONSISTENT_MEMBER_FIRST | tanda B; M hanya memecah member tie | keduanya nol → unresolved |

Member-first mengikuti aturan `MEMBER_FIRST` yang sudah ada pada notebook 12.
Magnitudo cluster-based tetap memakai distribusi evidence anggota dan
proyeksi 11 unit yang sama. Pada member tie dengan mode fallback, magnitude
adalah Balanced 1. Magnitudo direct memakai hitungan indikator peserta
secara eksplisit dan tetap merupakan ukuran behavioral, bukan skor ILS.

Ketiga operator berbasis cluster memakai mapping yang dibentuk dari train
untuk mengevaluasi test. Direct menerapkan evidence tetap pada fitur test
yang sudah dikategorikan dengan aturan train. ILS tidak dipakai untuk
menyetel threshold, memilih aturan, atau memilih run terbaik.


In [2]:
KONFIGURASI_27 = {
    'bootstrap_repetitions': 10000,
    'bootstrap_seed': 20261014,
    'save_member_comparisons': True,
    'force': False,
}


## Cakupan evaluasi dan pemakaian ulang

- **432 mapping utama** dan evidence-nya dibaca dari tahap 26.
- **864 mapping alternatif** dibentuk dari balance/magnitude tersimpan;
  partisi dan evidence indikator tidak dihitung ulang.
- Direct dihitung pada tiga konteks representasi: LMS full, LMS train/test,
  dan Pendikom train/test. Direct pada resample adalah subset full.
- **Pendikom 322/139**: 25 jalur unik = 8 initializer × 3 operator + 1 direct.
  Metode stokastik memakai seluruh 15 run, deterministik/direct satu run.
- **LMS**: 30 matched 80% resample per jalur = 750 baris evaluasi stabilitas;
  240 hasil primary dipakai kembali dari tahap 26.
- Target train/test, evidence decisions, konflik arah, prediksi per peserta,
  CI, tabel, dan gambar disimpan untuk analisis berikutnya.

Helper tahap 26 diverifikasi berdasarkan hash sumber.
Hanya pembaca artefak, validasi, representasi profil, metrik, serta fungsi
stabilitas yang digunakan; fungsi penggerak tahap 26 tidak dijalankan.
Tidak ada penulisan ke LIB maupun keluaran tahap 25/26.


In [3]:
UPSTREAM26_SOURCE = '"""Fixed-rule semantic evaluation from the immutable partition bank of stage 25."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport ast\nimport gzip\nimport hashlib\nimport importlib.metadata\nimport io\nimport itertools\nimport json\nimport os\nimport platform\nimport re\nimport sys\nimport uuid\nimport numpy as np\nimport pandas as pd\n\nSTAGE = \'26_Pengaruh_Initializer_Profil_Semantik\'\nUPSTREAM = \'25_Perbandingan_Initializer_Terintegrasi\'\nUP = \'OUTPUT/TAMBAHAN/\' + UPSTREAM\nUP_CONTROL = \'OUTPUT/TAMBAHAN/KONTROL/\' + UPSTREAM\nMETHODS = [\'Random\',\'Huang\',\'Cao\',\'Bai\',\'K-Modes++\',\'HR-2023\',\'CARE-KM\',\'BK-Modes\']\nSTOCHASTIC = {\'Random\',\'Huang\',\'K-Modes++\',\'HR-2023\'}\nORDER = [\'CARE-KM\',\'Cao\',\'Bai\',\'BK-Modes\',\'K-Modes++\',\'Random\',\'Huang\',\'HR-2023\']\nKEYS = [\'active_reflective\',\'sensing_intuitive\',\'visual_verbal\',\'sequential_global\']\nDIMS = [\'Active/Reflective\',\'Sensing/Intuitive\',\'Visual/Verbal\',\'Sequential/Global\']\nPOLES = [(\'Active\',\'Reflective\'),(\'Sensing\',\'Intuitive\'),(\'Visual\',\'Verbal\'),(\'Sequential\',\'Global\')]\nABBR = {\'Active\':\'Act.\',\'Reflective\':\'Ref.\',\'Sensing\':\'Sens.\',\'Intuitive\':\'Int.\',\n        \'Visual\':\'Vis.\',\'Verbal\':\'Ver.\',\'Sequential\':\'Seq.\',\'Global\':\'Gl.\'}\nLS_LOOKUP = {combo:\'LS%02d\' % (i+1) for i,combo in enumerate(itertools.product(*POLES))}\nMAIN = [\'content_visit\',\'content_stay\',\'selfass_visit_proxy\',\'selfass_stay_proxy\',\'exercise_visit_proxy\',\n        \'exercise_stay_proxy\',\'forum_visit\',\'forum_stay\',\'forum_post\',\'quiz_stay_results_proxy\',\n        \'navigation_overview_visit\',\'navigation_overview_stay\',\'return_to_overview_rate\']\nPEND = [\'content_visit\',\'content_stay\',\'selfass_visit\',\'selfass_twice_wrong\',\'exercise_visit\',\'exercise_stay\',\n        \'ques_detail\',\'ques_overview\',\'ques_facts\',\'ques_concepts\',\'ques_graphics\',\'ques_text\',\'ques_interpret\',\n        \'ques_develop\',\'quiz_revisions\',\'quiz_stay_results\',\'navigation_overview_visit\',\'navigation_overview_stay\']\nFULL = \'OUTPUT/PRAPROSES/01_Praproses_main_LMS/05_dataset_behavioral_categorical_skala.csv\'\nRULE_FILES = {\'LMS\':\'OUTPUT/EKSPERIMEN/SEMANTIK/11_Interpretasi_Semantik_FSLSM/08_semantic_evidence_matrix.csv\',\n              \'PENDIKOM\':\'OUTPUT/EKSPERIMEN/HOLDOUT/PENDIKOM/70_30_groundtruth_semantic_rules.csv\'}\nDEFAULT_CONFIG = {\'bootstrap_repetitions\':10000,\'bootstrap_seed\':20261014,\n                  \'save_member_comparisons\':True,\'force\':False}\nMETRIC_NAMES = [\'accuracy\',\'balanced_accuracy\',\'macro_f1\']\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef digest(value):\n    return sha(json.dumps(clean(value),sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False).encode(\'utf-8\'))\n\n\ndef clean(value):\n    if isinstance(value,dict): return {str(k):clean(v) for k,v in value.items()}\n    if isinstance(value,(list,tuple,np.ndarray)): return [clean(v) for v in value]\n    if isinstance(value,(np.integer,)): return int(value)\n    if isinstance(value,(float,np.floating)): return float(value) if np.isfinite(value) else None\n    if isinstance(value,np.bool_): return bool(value)\n    return value\n\n\ndef atomic(path, data):\n    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    temp = path.with_name(path.name+\'.\'+uuid.uuid4().hex[:8]+\'.tmp\')\n    temp.write_bytes(data); temp.replace(path)\n\n\ndef save_json(path, value):\n    atomic(path,json.dumps(clean(value),ensure_ascii=False,indent=2,allow_nan=False).encode(\'utf-8\'))\n\n\ndef save_csv(path, frame):\n    data = frame.to_csv(index=False,lineterminator=\'\\n\',float_format=\'%.17g\').encode(\'utf-8\')\n    atomic(path,gzip.compress(data,mtime=0) if str(path).endswith(\'.gz\') else data)\n\n\ndef read_checkpoint(path):\n    try:\n        obj = json.loads(Path(path).read_text(encoding=\'utf-8\'))\n        return obj if isinstance(obj,dict) else {}\n    except (OSError,ValueError,UnicodeError): return {}\n\n\nclass Files:\n    def __init__(self, root):\n        self.root = Path(root).resolve(); self.cache = {}\n\n    def path(self, relative):\n        path = (self.root / relative).resolve()\n        if not path.is_relative_to(self.root): raise ValueError(\'Lokasi di luar proyek: \'+relative)\n        return path\n\n    def read(self, relative):\n        path = self.path(relative); before = path.stat()\n        stamp = (before.st_size,before.st_mtime_ns,before.st_ctime_ns)\n        if relative in self.cache and self.cache[relative][0] == stamp: return self.cache[relative][1]\n        data = path.read_bytes(); after = path.stat()\n        if stamp != (after.st_size,after.st_mtime_ns,after.st_ctime_ns):\n            raise RuntimeError(\'Berkas berubah ketika dibaca: \'+relative)\n        self.cache[relative] = (stamp,data); return data\n\n    def hash(self, relative): return sha(self.read(relative))\n\n    def frame(self, relative, strings=False):\n        data = self.read(relative)\n        if relative.endswith(\'.gz\'): data = gzip.decompress(data)\n        return pd.read_csv(io.BytesIO(data),dtype=str if strings else None,\n                           keep_default_na=not strings,float_precision=\'round_trip\')\n\n\ndef runtime():\n    packages = {}\n    for name in [\'numpy\',\'pandas\',\'scipy\',\'sklearn\',\'matplotlib\']:\n        mod = sys.modules.get(name)\n        packages[name] = getattr(mod,\'__version__\',None) or importlib.metadata.version(\'scikit-learn\' if name==\'sklearn\' else name)\n    return {\'python\':platform.python_version(),\'packages\':packages}\n\n\ndef notebook_hash(files, stem, tag, variable):\n    nb = json.loads(files.read(\'PYTHON/\'+stem+\'.ipynb\').decode(\'utf-8-sig\')); code=[]\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']!=\'code\': continue\n        source=\'\'.join(cell[\'source\'])\n        if tag in cell.get(\'metadata\',{}).get(\'tags\',[]):\n            tree=ast.parse(source)\n            if (len(tree.body)!=1 or not isinstance(tree.body[0],ast.Assign)\n                or len(tree.body[0].targets)!=1 or not isinstance(tree.body[0].targets[0],ast.Name)\n                or tree.body[0].targets[0].id!=variable or not isinstance(ast.literal_eval(tree.body[0].value),dict)):\n                raise RuntimeError(\'Sel konfigurasi hanya boleh berisi literal \'+variable)\n        else: code.append(source)\n    return digest(code)\n\n\ndef expected_rules(dataset):\n    rows=[]\n    def add(features,dim,low,high):\n        for f in features.split(): rows.append((f,dim,low,high))\n    if dataset==\'LMS\':\n        add(\'content_visit content_stay selfass_stay_proxy forum_visit quiz_stay_results_proxy\',DIMS[0],*POLES[0])\n        add(\'selfass_visit_proxy exercise_visit_proxy exercise_stay_proxy forum_post\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay\',DIMS[1],*POLES[1])\n        add(\'selfass_visit_proxy selfass_stay_proxy exercise_visit_proxy quiz_stay_results_proxy\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit forum_visit forum_stay forum_post\',DIMS[2],*POLES[2])\n        add(\'navigation_overview_visit navigation_overview_stay return_to_overview_rate\',DIMS[3],*POLES[3])\n    else:\n        add(\'selfass_visit selfass_twice_wrong exercise_visit exercise_stay\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay quiz_stay_results\',DIMS[0],*POLES[0])\n        add(\'selfass_visit exercise_visit ques_detail ques_facts quiz_revisions quiz_stay_results\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit content_stay ques_concepts ques_develop\',DIMS[1],*POLES[1])\n        add(\'ques_graphics\',DIMS[2],*POLES[2][::-1]); add(\'content_visit ques_text\',DIMS[2],*POLES[2])\n        add(\'ques_detail\',DIMS[3],*POLES[3][::-1])\n        add(\'ques_overview ques_interpret ques_develop navigation_overview_visit navigation_overview_stay\',DIMS[3],*POLES[3])\n    return sorted(rows)\n\n\ndef check_config(configuration):\n    if set(configuration)-set(DEFAULT_CONFIG): raise ValueError(\'Konfigurasi tidak dikenal.\')\n    cfg=dict(DEFAULT_CONFIG);cfg.update(configuration)\n    for key in [\'bootstrap_repetitions\',\'bootstrap_seed\']:\n        if isinstance(cfg[key],bool) or not isinstance(cfg[key],int) or cfg[key]<0: raise ValueError(key+\' tidak valid.\')\n    if cfg[\'bootstrap_repetitions\']<100: raise ValueError(\'Gunakan sedikitnya 100 bootstrap untuk diagnostik; default publikasi 10000.\')\n    for key in [\'save_member_comparisons\',\'force\']:\n        if not isinstance(cfg[key],bool): raise ValueError(key+\' harus True/False.\')\n    return cfg\n\n\ndef input_setup(files):\n    required = [UP+\'/PROTOKOL/02_konfigurasi.json\',UP_CONTROL+\'/status_eksekusi.json\',\n                FULL,*RULE_FILES.values(),\'LIB/pipeline_data.py\',\'LIB/pendikom_ils_labels.csv\',\n                \'OUTPUT/SPLIT/LMS/train_categorizer_state.json\',\n                \'OUTPUT/SPLIT/LMS/70_30_main_lms_train_only_thresholds.csv\',\n                \'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',\n                \'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\',\n                \'PYTHON/\'+UPSTREAM+\'.ipynb\',\'PYTHON/\'+STAGE+\'.ipynb\']\n    for ds in [\'LMS\',\'PENDIKOM\']:\n        required += [f\'OUTPUT/SPLIT/{ds}/{part}/categorical.csv\' for part in [\'TRAIN\',\'TEST\']]\n    required += [f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{r:02d}.csv\' for r in range(30)]\n    missing=[p for p in required if not files.path(p).is_file()]\n    if missing: raise FileNotFoundError(\'Masukan belum tersedia. Selesaikan tahap 25 dan pertahankan hasil 01–24:\\n\'+\'\\n\'.join(missing[:20]))\n    status=json.loads(files.read(UP_CONTROL+\'/status_eksekusi.json\'))\n    if status.get(\'status\')!=\'selesai\' or status.get(\'full_protocol\') is not True or status.get(\'n_valid\')!=3272:\n        raise RuntimeError(\'Tahap 25 belum selesai dengan protokol lengkap dan seluruh run valid.\')\n    protocol=json.loads(files.read(UP+\'/PROTOKOL/02_konfigurasi.json\'))\n    if protocol[\'constants\'] != {\'FROZEN_CARE_CONFIG\':{\'candidate_rule\':\'separation\',\'candidate_budget\':16,\'min_gain\':0.0},\'K\':16,\'MAX_ITER\':100,\'SI_SAMPLE_SIZE\':2000}:\n        raise RuntimeError(\'Konfigurasi tahap 25 berbeda dari protokol yang ditetapkan.\')\n    source_code=protocol[\'code\']\n    for relative,checksum in source_code[\'project_libraries\'].items():\n        if files.hash(relative)!=checksum: raise RuntimeError(\'Pustaka sumber tahap 25 berubah: \'+relative)\n        required.append(relative)\n    if notebook_hash(files,UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\') != source_code[\'notebook_code\']:\n        raise RuntimeError(\'Kode notebook 25 berubah setelah eksperimen. Gunakan notebook sumber yang sesuai.\')\n    literals={}\n    nb=json.loads(files.read(\'PYTHON/\'+UPSTREAM+\'.ipynb\'))\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']==\'code\':\n            for node in ast.parse(\'\'.join(cell[\'source\'])).body:\n                if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and isinstance(node.value,ast.Constant):\n                    literals[node.targets[0].id]=node.value.value\n    for name,var in [(\'initializer\',\'BK_MODES_SOURCE\'),(\'verification\',\'VERIFICATION_SOURCE\'),(\'comparison\',\'COMPARISON_SOURCE\')]:\n        if sha(literals[var].encode(\'utf-8\'))!=source_code[\'embedded_sources\'][name]: raise RuntimeError(\'Sumber tertanam tahap 25 tidak sesuai.\')\n    feature_state={}\n    for node in ast.parse(files.read(\'LIB/pipeline_data.py\').decode(\'utf-8-sig\')).body:\n        if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and node.targets[0].id in [\'MAIN_FEATURES\',\'PENDIKOM_FEATURES\']:\n            feature_state[node.targets[0].id]=ast.literal_eval(node.value)\n    if feature_state[\'MAIN_FEATURES\']!=MAIN or [x for x in feature_state[\'PENDIKOM_FEATURES\'] if x!=\'navigation_skip\']!=PEND:\n        raise RuntimeError(\'Urutan fitur tidak sesuai protokol.\')\n    audit=files.frame(\'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',True)\n    if audit.loc[audit.used.str.lower().eq(\'true\'),\'feature\'].tolist()!=PEND: raise RuntimeError(\'Mask fitur train Pendikom berubah.\')\n    rules={}\n    for ds,path in RULE_FILES.items():\n        frame=files.frame(path,True)\n        cols=[\'feature\',\'dimension\',\'low_supports\',\'high_supports\']\n        if sorted(map(tuple,frame[cols].to_numpy())) != expected_rules(ds): raise RuntimeError(\'Evidence rules berbeda dari aturan utama: \'+ds)\n        if \'moderate_supports\' in frame and not frame.moderate_supports.eq(\'Neutral\').all(): raise RuntimeError(\'Evidence moderate harus netral.\')\n        rules[ds]=frame[cols]\n    watched={p:files.hash(p) for p in set(required) if not p.endswith(\'.ipynb\') and not p.endswith(\'status_eksekusi.json\')}\n    return protocol,rules,watched\n\n\ndef contexts(files):\n    def make(scope,ds,frame,features,idcol,source,rep=None,idx=None,test=None):\n        if frame[idcol].duplicated().any(): raise RuntimeError(\'Identitas peserta tidak unik: \'+scope)\n        X=frame[features].to_numpy(dtype=object)\n        if not np.isin(X,[\'1\',\'2\',\'3\']).all(): raise RuntimeError(\'Kategori di luar 1/2/3: \'+scope)\n        indices=np.arange(len(X),dtype=np.int64) if idx is None else idx\n        ids=frame[idcol].tolist();testids=test[idcol].tolist() if test is not None else []\n        if len(set(testids))!=len(testids) or set(ids)&set(testids): raise RuntimeError(\'Train/test tumpang tindih atau ID test berulang.\')\n        Xt=test[features].to_numpy(dtype=object) if test is not None else None\n        matrix=digest({\'features\':features,\'rows\':X.tolist(),\'ids\':ids,\'indices\':indices.tolist(),\n                       \'test_rows\':Xt.tolist() if Xt is not None else None,\'test_ids\':testids,\'classes\':None})\n        return {\'scope\':scope,\'dataset\':ds,\'X\':X,\'ids\':ids,\'indices\':indices,\'test_X\':Xt,\'test_ids\':testids,\n                \'features\':features,\'sources\':source,\'rep\':rep,\'matrix_sha256\':matrix,\'test_frame\':test}\n    frame=files.frame(FULL,True)\n    if frame.shape!=(6658,14) or frame.columns.tolist()!=[\'user\']+MAIN: raise RuntimeError(\'Matriks full LMS berbeda.\')\n    yield make(\'LMS_FULL\',\'LMS\',frame,MAIN,\'user\',[FULL])\n    for rep in range(30):\n        path=f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{rep:02d}.csv\'\n        idx=files.frame(path)[\'source_row_index\'].to_numpy(np.int64)\n        if len(idx)!=5326 or len(np.unique(idx))!=5326 or np.any(idx<0) or np.any(idx>=6658): raise RuntimeError(\'Indeks resample tidak valid.\')\n        yield make(\'LMS_RESAMPLING\',\'LMS\',frame.iloc[idx],MAIN,\'user\',[FULL,path],rep,idx)\n    for ds,ntr,nte,features,idcol in [(\'LMS\',4660,1998,MAIN,\'user\'),(\'PENDIKOM\',322,139,PEND,\'userid\')]:\n        paths=[f\'OUTPUT/SPLIT/{ds}/{part}/categorical.csv\' for part in [\'TRAIN\',\'TEST\']]\n        tr,te=[files.frame(p,True) for p in paths]\n        if len(tr)!=ntr or len(te)!=nte: raise RuntimeError(\'Ukuran train/test berbeda: \'+ds)\n        if \'split\' in tr and (not tr.split.str.lower().eq(\'train\').all() or not te.split.str.lower().eq(\'test\').all()): raise RuntimeError(\'Kolom split tidak sesuai.\')\n        yield make(ds+\'_TRAIN\',ds,tr,features,idcol,paths,test=te)\n\n\ndef job_identity(context,method,run):\n    return {\'scope\':context[\'scope\'],\'dataset\':context[\'dataset\'],\'rep\':context[\'rep\'],\'run\':run,\'method\':method}\n\n\ndef upstream_job(files,protocol,context,method,run):\n    identity=job_identity(context,method,run);key=digest(identity)[:24]\n    relative=UP_CONTROL+\'/JOBS/\'+key+\'.json\'\n    if not files.path(relative).is_file(): raise FileNotFoundError(\'Partisi tahap 25 belum tersedia: \'+str(identity))\n    record=json.loads(files.read(relative));checksum=record.pop(\'record_sha256\',\'\')\n    if digest(record)!=checksum or record.get(\'status\')!=\'selesai\' or record.get(\'identity\')!=identity:\n        raise RuntimeError(\'Catatan partisi tahap 25 tidak valid: \'+relative)\n    p=record[\'parameters\'];row=record[\'row\']\n    if record[\'signature\']!=digest({\'parameters\':p,\'code\':protocol[\'code\'],\'runtime\':protocol[\'runtime\']}): raise RuntimeError(\'Signature tahap 25 tidak sesuai protokol.\')\n    expected_seed=(6161600+context[\'rep\']) if context[\'rep\'] is not None else 4161600+run\n    if (p[\'matrix_sha256\']!=context[\'matrix_sha256\'] or p[\'k\']!=16 or p[\'seed\']!=expected_seed\n        or p[\'max_iter\']!=100 or p[\'care_config\']!=protocol[\'constants\'][\'FROZEN_CARE_CONFIG\']\n        or not p[\'needs_partition\'] or row.get(\'valid\') is not True or not row.get(\'partition_available\')):\n        raise RuntimeError(\'Isi matriks/parameter/partisi tahap 25 tidak sesuai: \'+str(identity))\n    watched={relative:files.hash(relative)}\n    for path,expected in record[\'outputs\'].items():\n        if not path.startswith(UP+\'/BANK/\'): raise RuntimeError(\'Lokasi bank tahap 25 tidak sesuai.\')\n        if files.hash(path)!=expected: raise RuntimeError(\'Artefak tahap 25 berubah. Pulihkan/jalankan ulang 25: \'+path)\n        watched[path]=expected\n    def find(filename):\n        found=[p for p in record[\'outputs\'] if p.endswith(\'/\'+filename)]\n        if len(found)!=1: raise RuntimeError(\'Artefak wajib tidak unik: \'+filename)\n        return found[0]\n    modes=files.frame(find(\'final_modes.csv\'),True)\n    if modes.columns.tolist()!=[\'cluster_id\']+context[\'features\'] or modes.cluster_id.astype(int).tolist()!=list(range(16)):\n        raise RuntimeError(\'Skema/urutan mode akhir tidak sesuai.\')\n    modes=modes[context[\'features\']].to_numpy(dtype=object)\n    if not np.isin(modes,[\'1\',\'2\',\'3\']).all(): raise RuntimeError(\'Kategori mode tidak valid.\')\n    assignment=files.frame(find(\'train_assignments.csv\'),True)\n    if assignment.identity.tolist()!=context[\'ids\'] or not np.array_equal(assignment.source_row_index.astype(np.int64),context[\'indices\']):\n        raise RuntimeError(\'Identitas/urutan anggota tidak sesuai matriks.\')\n    labels=assignment.cluster_id.to_numpy(int)\n    if not np.isin(labels,np.arange(16)).all() or (np.bincount(labels,minlength=16)==0).any(): raise RuntimeError(\'Keanggotaan cluster tidak valid.\')\n    cost=int(np.sum(context[\'X\']!=modes[labels]))\n    if cost!=int(row[\'final_cost\']): raise RuntimeError(\'Cost anggota/mode berbeda dari hasil tahap 25.\')\n    testlabels=None\n    if context[\'test_X\'] is not None:\n        test=files.frame(find(\'test_assignments.csv\'),True)\n        if test.identity.tolist()!=context[\'test_ids\']: raise RuntimeError(\'Urutan identitas test berbeda.\')\n        testlabels=test.cluster_id.to_numpy(int)\n        distance=np.zeros((len(testlabels),16),dtype=np.int64)\n        for j in range(len(context[\'features\'])): distance += context[\'test_X\'][:,j,None]!=modes[None,:,j]\n        if not np.array_equal(testlabels,np.argmin(distance,axis=1)) or not np.array_equal(test.distance.astype(int),distance.min(axis=1)):\n            raise RuntimeError(\'Assignment test berbeda dari mode train yang dibekukan.\')\n    return {\'record\':record,\'checksum\':checksum,\'watched\':watched,\'modes\':modes,\'labels\':labels,\'test_labels\':testlabels,\'identity\':identity}\n\n\ndef interpret(X,modes,labels,features,rules):\n    """Same mode-first rule and member-derived 11-unit magnitude as stages 11/13/22."""\n    sizes=np.bincount(labels,minlength=16);mapping=[];evidence=[];fpos={f:i for i,f in enumerate(features)}\n    prepared=[]\n    for dim,poles in zip(DIMS,POLES):\n        rr=rules.loc[rules.dimension.eq(dim)];cols=np.array([fpos[f] for f in rr.feature]);low=np.array([1 if v==poles[0] else -1 for v in rr.low_supports])\n        high=np.array([1 if v==poles[0] else -1 for v in rr.high_supports])\n        data=X[:,cols];vote=np.where(data==\'1\',low,np.where(data==\'3\',high,0))\n        mv=modes[:,cols];modevote=np.where(mv==\'1\',low,np.where(mv==\'3\',high,0))\n        prepared.append((rr,vote,modevote))\n    for cid in range(16):\n        item={\'cluster_id\':cid,\'cluster_name\':f\'C{cid+1:02d}\',\'cluster_size\':int(sizes[cid])};poles_selected=[];sources=[];conflicts=0\n        for di,(dim,poles,key) in enumerate(zip(DIMS,POLES,KEYS)):\n            rr,vote,modevote=prepared[di];mv=modevote[cid];nv=vote[labels==cid]\n            mp,mn,mz=[int(np.sum(mv==v)) for v in [1,-1,0]]\n            pos,neg,neu=[int(np.sum(nv==v)) for v in [1,-1,0]]\n            ms,ns=mp-mn,pos-neg;selected=1 if ms>0 else -1 if ms<0 else 1 if ns>0 else -1 if ns<0 else 0\n            source=\'MODE\' if ms else \'MEMBER\' if ns else \'BALANCED\';sources.append(source)\n            total=pos+neg+neu\n            units=min(11,max(0,int(np.floor(11*(pos+.5*neu)/total+.5)))) if total else None\n            raw=2*units-11 if units is not None else None;mag=abs(raw) if raw is not None else None\n            category=\'Balanced\' if mag in (1,3) else \'Moderate preference\' if mag in (5,7) else \'Strong preference\' if mag in (9,11) else \'Unresolved\'\n            pole=poles[0] if selected==1 else poles[1] if selected==-1 else None\n            short=ABBR[pole] if pole else \'Balanced\';poles_selected.append(pole)\n            conflict=bool(selected and ns and selected!=np.sign(ns));conflicts+=conflict\n            item[key]=short;item[key+\'_score\']=selected*mag if selected and mag is not None else np.nan\n            item[key+\'_category\']=category;item[key+\'_code\']=short+str(mag) if selected else \'Balanced\'\n            item[key+\'_resolved\']=bool(selected)\n            evidence.append({\'cluster_id\':cid,\'dimension\':dim,\'cluster_size\':int(sizes[cid]),\'mapped_features\':len(rr),\n                \'mode_positive\':mp,\'mode_negative\':mn,\'mode_neutral\':mz,\'mode_score\':ms,\'member_positive\':pos,\'member_negative\':neg,\n                \'member_neutral\':neu,\'member_score\':ns,\'selected_pole\':short,\'decision_source\':source,\n                \'positive_adjusted_evidence\':pos+.5*neu,\'negative_adjusted_evidence\':neg+.5*neu,\n                \'positive_11eq_units\':units,\'negative_11eq_units\':11-units if units is not None else None,\n                \'member_11eq_raw_score\':raw,\'preference_11eq_magnitude\':mag,\'preference_category\':category,\n                \'mode_member_consistency\':not conflict,\'strong_mode_member_conflict\':conflict and mag in (9,11)})\n        resolved=all(p is not None for p in poles_selected)\n        item.update(ls_class=LS_LOOKUP[tuple(poles_selected)] if resolved else \'AMBIGUOUS\',\n                    semantic_label=\'-\'.join(item[k] for k in KEYS),preference_signature=\'|\'.join(item[k+\'_code\'] for k in KEYS),\n                    mapping_status=\'RESOLVED\' if resolved else \'UNRESOLVED\',mode_dimensions=sources.count(\'MODE\'),\n                    member_dimensions=sources.count(\'MEMBER\'),balanced_dimensions=sources.count(\'BALANCED\'),mode_member_disagreements=conflicts)\n        mapping.append(item)\n    return pd.DataFrame(mapping),pd.DataFrame(evidence)\n\n\ndef profiles(mapping,labels,ids,indices=None):\n    cols=KEYS+[k+\'_score\' for k in KEYS]+[\'ls_class\',\'mapping_status\']\n    frame=mapping.set_index(\'cluster_id\').loc[labels,cols].reset_index()\n    frame.insert(0,\'identity\',ids)\n    if indices is not None: frame.insert(1,\'source_row_index\',indices)\n    return frame\n\n\ndef numeric_profiles(frame):\n    return np.column_stack([np.where(frame[k].eq(ABBR[p[0]]),1,np.where(frame[k].eq(ABBR[p[1]]),-1,0)) for k,p in zip(KEYS,POLES)])\n\n\ndef cache_read(files,path,signature,force=False):\n    record=read_checkpoint(path)\n    checksum=record.pop(\'record_sha256\',\'\')\n    try: valid=digest(record)==checksum\n    except (TypeError,ValueError): valid=False\n    if force or not valid or record.get(\'signature\')!=signature or record.get(\'status\')!=\'selesai\': return None\n    if not all(files.path(p).is_file() and files.hash(p)==h for p,h in record.get(\'outputs\',{}).items()): return None\n    return record\n\n\ndef cache_write(files,path,signature,outputs,extra):\n    record={\'status\':\'selesai\',\'signature\':signature,\'outputs\':{str(p.relative_to(files.root)).replace(\'\\\\\',\'/\'):sha(p.read_bytes()) for p in outputs},**extra}\n    record[\'record_sha256\']=digest(record);save_json(path,record);return record\n\n\ndef semantic_job(files,output,control,context,upstream,rules,code_state,cfg):\n    identity=upstream[\'identity\'];key=digest(identity)[:24]\n    signature=digest({\'code\':code_state,\'upstream_record_sha256\':upstream[\'checksum\'],\n                      \'matrix_sha256\':context[\'matrix_sha256\'],\'evidence\':rules.to_dict(\'records\'),\'rule\':\'mode-first-member-tie-11eq-v1\'})\n    checkpoint=control/\'PROFIL\'/f\'{key}.json\'\n    record=cache_read(files,checkpoint,signature,cfg[\'force\'])\n    folder=output/\'BANK\'/identity[\'scope\']/identity[\'dataset\']/(\'r%02d\' % (identity[\'rep\'] if identity[\'rep\'] is not None else identity[\'run\']))/re.sub(r\'[^A-Za-z0-9]+\',\'_\',identity[\'method\'].replace(\'++\',\'_PP\')).strip(\'_\')\n    mapping_path=folder/\'01_cluster_profiles.csv\';evidence_path=folder/\'02_cluster_dimension_evidence.csv\'\n    if record:\n        mapping=files.frame(str(mapping_path.relative_to(files.root)).replace(\'\\\\\',\'/\'));evidence=files.frame(str(evidence_path.relative_to(files.root)).replace(\'\\\\\',\'/\'))\n        disposition=\'CACHE\'\n    else:\n        mapping,evidence=interpret(context[\'X\'],upstream[\'modes\'],upstream[\'labels\'],context[\'features\'],rules)\n        save_csv(mapping_path,mapping);save_csv(evidence_path,evidence);paths=[mapping_path,evidence_path]\n        if context[\'test_X\'] is not None:\n            for filename,frame in [(\'03_train_targets.csv\',profiles(mapping,upstream[\'labels\'],context[\'ids\'],context[\'indices\'])),\n                                   (\'04_test_targets.csv\',profiles(mapping,upstream[\'test_labels\'],context[\'test_ids\']))]:\n                path=folder/filename;save_csv(path,frame);paths.append(path)\n        record=cache_write(files,checkpoint,signature,paths,{\'identity\':identity,\'upstream_artifacts\':upstream[\'watched\']})\n        disposition=\'MAP\'\n    weighted=sum(mapping.cluster_size*mapping.mapping_status.ne(\'RESOLVED\'))/len(context[\'X\'])\n    summary={**identity,\'n_learners\':len(context[\'X\']),\'n_clusters\':16,\'n_resolved_profiles\':mapping.loc[mapping.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique(),\n             \'unresolved_learner_rate\':weighted,\'mode_member_conflicts\':int((~evidence.mode_member_consistency.astype(bool)).sum()),\n             \'strong_conflicts\':int(evidence.strong_mode_member_conflict.astype(bool).sum()),\'mapping_file\':str(mapping_path.relative_to(files.root)).replace(\'\\\\\',\'/\'),\n             \'evidence_file\':str(evidence_path.relative_to(files.root)).replace(\'\\\\\',\'/\')}\n    return mapping,evidence,record,summary,disposition\n\n\ndef count_metrics(counts):\n    """2 x 3 confusion counts; column 3 is abstention, never dropped."""\n    a,b,c,d,e,f=np.moveaxis(np.asarray(counts,float),-1,0)\n    pos=a+b+c;neg=d+e+f;total=pos+neg\n    with np.errstate(divide=\'ignore\',invalid=\'ignore\'):\n        accuracy=(a+e)/total;ba=.5*(a/pos+e/neg)\n        denp=2*a+d+b+c;denn=2*e+b+d+f\n        fp=np.divide(2*a,denp,out=np.zeros_like(a),where=denp>0)\n        fn=np.divide(2*e,denn,out=np.zeros_like(e),where=denn>0)\n    return np.stack([accuracy,ba,.5*(fp+fn)],axis=-1)\n\n\ndef confusion_masks(truth,prediction):\n    # prediction: run x learner x dimension; output learner x run x dimension x six cells.\n    t=truth[None,:,:];p=prediction\n    conditions=[(t==1)&(p==1),(t==1)&(p==-1),(t==1)&(p==0),\n                (t==-1)&(p==1),(t==-1)&(p==-1),(t==-1)&(p==0)]\n    return np.stack(conditions,axis=-1).transpose(1,0,2,3).astype(float)\n\n\ndef ils_reference(files,context):\n    path=\'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\';ref=files.frame(path,True)\n    test=context[\'test_frame\']\n    if ref.userid.tolist()!=context[\'test_ids\'] or ref.courseid.tolist()!=test.courseid.tolist(): raise RuntimeError(\'Reference ILS tidak sejajar dengan peserta test.\')\n    original=files.frame(\'LIB/pendikom_ils_labels.csv\',True).set_index([\'userid\',\'courseid\'])\n    original=original.loc[list(zip(ref.userid,ref.courseid))].reset_index()\n    cols=[\'ils_\'+k+\'_code\' for k in KEYS]+[\'ils_ls_class\']\n    if not original[cols].reset_index(drop=True).equals(ref[cols].reset_index(drop=True)) or not original.split.str.lower().eq(\'test\').all():\n        raise RuntimeError(\'Reference ILS test berbeda dari label independen yang dibekukan.\')\n    truth=[]\n    for key,poles in zip(KEYS,POLES):\n        column=ref[\'ils_\'+key+\'_code\'];first=column.str.startswith(ABBR[poles[0]]);second=column.str.startswith(ABBR[poles[1]])\n        if not (first^second).all(): raise RuntimeError(\'Label arah ILS tidak dikenal.\')\n        values=np.where(first,1,-1)\n        if len(np.unique(values))!=2: raise RuntimeError(\'Dua kutub reference harus terwakili untuk metrik BA.\')\n        truth.append(values)\n    return ref,np.column_stack(truth)\n\n\ndef evaluate_ils(files,output,control,predictions,reference,truth,code_state,cfg):\n    folder=output/\'ILS\';checkpoint=control/\'evaluasi_ils.json\'\n    signature=digest({\'code\':code_state,\'truth\':truth.tolist(),\'ids\':reference.userid.tolist(),\n        \'predictions\':{m:sha(np.asarray(predictions[m],dtype=np.int8).tobytes()) for m in METHODS},\n        \'bootstrap_repetitions\':cfg[\'bootstrap_repetitions\'],\'bootstrap_seed\':cfg[\'bootstrap_seed\'],\n        \'unit\':\'learner; all four dimensions and all runs/methods remain paired\'})\n    saved=cache_read(files,checkpoint,signature,cfg[\'force\'])\n    if saved:\n        print(\'Evaluasi ILS/CI: gunakan hasil yang isinya terverifikasi sama.\',flush=True)\n        return files.frame(str((folder/\'04_ringkasan_overall.csv\').relative_to(files.root)).replace(\'\\\\\',\'/\')),True\n    from threadpoolctl import threadpool_limits\n    n=len(truth);nr=cfg[\'bootstrap_repetitions\'];rng=np.random.default_rng(cfg[\'bootstrap_seed\'])\n    masks={m:confusion_masks(truth,np.asarray(predictions[m],dtype=np.int8)) for m in METHODS}\n    counts={m:masks[m].sum(axis=0) for m in METHODS}\n    points={m:count_metrics(counts[m]) for m in METHODS}\n    boot=np.empty((nr,len(METHODS),5,3),dtype=np.float64)\n    valid=np.zeros(nr,dtype=bool)\n    print(\'CI ILS:\',nr,\'bootstrap berpasangan pada\',n,\'peserta.\',flush=True)\n    with threadpool_limits(limits=1):\n        for start in range(0,nr,250):\n            stop=min(start+250,nr);weights=rng.multinomial(n,np.full(n,1/n),size=stop-start).astype(float)\n            positive=weights@(truth==1);negative=n-positive\n            valid[start:stop]=((positive>0)&(negative>0)).all(axis=1)\n            for mi,method in enumerate(METHODS):\n                source=masks[method];shape=source.shape\n                values=(weights@source.reshape(n,-1)).reshape(stop-start,shape[1],4,6)\n                metrics=count_metrics(values).mean(axis=1)\n                boot[start:stop,mi,:4,:]=metrics;boot[start:stop,mi,4,:]=metrics.mean(axis=1)\n    if valid.sum()<100: raise RuntimeError(\'Bootstrap valid kurang dari 100; interval belum dapat dihitung.\')\n    boot=boot[valid];paths=[];raw_rows=[];cm_rows=[];ref_rows=[];summary=[];overall=[];pred_rows=[]\n    actual_counts=np.stack([(truth==1).sum(axis=0),(truth==-1).sum(axis=0)],axis=-1)\n    for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n        for ci,pole in enumerate(poles): ref_rows.append({\'dimension\':key,\'reference_pole\':pole,\'n\':int(actual_counts[di,ci]),\'n_test\':n})\n    for mi,method in enumerate(METHODS):\n        arr=np.asarray(predictions[method],dtype=np.int8);point=points[method];nrun=len(arr)\n        for run in range(nrun):\n            for li,identity in enumerate(reference.userid):\n                row={\'method\':method,\'run\':run,\'seed\':4161600+run,\'identity\':identity,\'courseid\':reference.courseid.iloc[li],\n                     \'reference_ls_class\':reference.ils_ls_class.iloc[li]}\n                full=[]\n                for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n                    row[\'reference_\'+key]=poles[0] if truth[li,di]==1 else poles[1]\n                    value=arr[run,li,di];pole=poles[0] if value==1 else poles[1] if value==-1 else None\n                    row[\'predicted_\'+key]=pole or \'UNRESOLVED\';full.append(pole)\n                row[\'predicted_ls_class\']=LS_LOOKUP[tuple(full)] if all(full) else \'AMBIGUOUS\';pred_rows.append(row)\n            for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n                cc=counts[method][run,di];a,b,c,d,e,f=cc;resolved=n-c-f\n                raw_rows.append({\'method\':method,\'run\':run,\'dimension\':key,\'n_test\':n,\'reference_first\':int(actual_counts[di,0]),\n                                 \'reference_second\':int(actual_counts[di,1]),\'n_resolved\':int(resolved),\'n_unresolved\':int(c+f),\n                                 \'coverage\':resolved/n,\'resolved_only_accuracy\':(a+e)/resolved if resolved else np.nan,\n                                 **dict(zip(METRIC_NAMES,point[run,di]))})\n                for ci,num in enumerate(cc):\n                    cm_rows.append({\'method\':method,\'run\':run,\'dimension\':key,\'reference_pole\':poles[ci//3],\n                                    \'predicted_pole\':(poles+(\'UNRESOLVED\',))[ci%3],\'n\':int(num)})\n        for di,key in enumerate(KEYS+[\'mean_four_dimensions\']):\n            values=point[:,di,:] if di<4 else point.mean(axis=1)\n            row={\'method\':method,\'dimension\':key,\'n_runs\':nrun,\'n_test\':n,\n                 \'bootstrap_valid\':int(valid.sum()),\'bootstrap_requested\':nr,\n                 \'coverage_mean\':float((arr[:,:,di]!=0).mean()) if di<4 else float((arr!=0).mean()),\n                 \'complete_profile_coverage\':float((arr!=0).all(axis=2).mean())}\n            for ji,metric in enumerate(METRIC_NAMES):\n                low,high=np.quantile(boot[:,mi,di,ji],[.025,.975])\n                row[metric+\'_mean\']=float(values[:,ji].mean());row[metric+\'_sd_runs\']=float(values[:,ji].std(ddof=1)) if nrun>1 else np.nan\n                row[metric+\'_ci95_low\']=low;row[metric+\'_ci95_high\']=high\n            (summary if di<4 else overall).append(row)\n    differences=[];care=METHODS.index(\'CARE-KM\')\n    for mi,method in enumerate(METHODS):\n        if method==\'CARE-KM\': continue\n        for di,key in enumerate(KEYS+[\'mean_four_dimensions\']):\n            for ji,metric in enumerate(METRIC_NAMES):\n                delta=boot[:,care,di,ji]-boot[:,mi,di,ji];lo,hi=np.quantile(delta,[.025,.975])\n                carepoint=points[\'CARE-KM\'][:,di,ji].mean() if di<4 else points[\'CARE-KM\'][:,:,ji].mean()\n                basepoint=points[method][:,di,ji].mean() if di<4 else points[method][:,:,ji].mean()\n                row={\'baseline\':method,\'dimension\':key,\'metric\':metric,\'care_minus_baseline\':carepoint-basepoint,\n                     \'paired_ci95_low\':lo,\'paired_ci95_high\':hi,\'positive_favors\':\'CARE-KM\',\'n_learners\':n,\n                     \'interval\':\'paired learner percentile bootstrap; individual 95% intervals\'}\n                if di==4 and metric==\'balanced_accuracy\':\n                    low,high=np.quantile(delta,[.05/(2*7),1-.05/(2*7)])\n                    row.update(primary_family=\'seven overall mean-BA contrasts\',bonferroni_family95_low=low,bonferroni_family95_high=high)\n                differences.append(row)\n    outputs=[(\'01_prediksi_per_peserta.csv\',pd.DataFrame(pred_rows)),(\'02_metrik_per_dimensi_per_run.csv\',pd.DataFrame(raw_rows)),\n             (\'03_ringkasan_per_dimensi.csv\',pd.DataFrame(summary)),(\'04_ringkasan_overall.csv\',pd.DataFrame(overall)),\n             (\'05_confusion_matrices.csv\',pd.DataFrame(cm_rows)),(\'07_perbedaan_berpasangan_vs_care.csv\',pd.DataFrame(differences)),\n             (\'08_reference_counts.csv\',pd.DataFrame(ref_rows))]\n    for filename,frame in outputs:\n        path=folder/filename;save_csv(path,frame);paths.append(path)\n    buffer=io.BytesIO();np.savez_compressed(buffer,metrics=boot,methods=np.asarray(METHODS),dimensions=np.asarray(KEYS+[\'mean_four_dimensions\']),\n                                          metric_names=np.asarray(METRIC_NAMES),valid_draws=valid,seed=cfg[\'bootstrap_seed\'])\n    path=folder/\'06_bootstrap_paired_metrics.npz\';atomic(path,buffer.getvalue());paths.append(path)\n    path=folder/\'09_protokol_evaluasi.json\'\n    save_json(path,{\'primary_metric\':\'mean balanced accuracy across four dimensions\',\'n_learners\':n,\'n_bootstrap_valid\':int(valid.sum()),\n        \'seed\':cfg[\'bootstrap_seed\'],\'unit\':\'learner; all dimensions, runs, and methods paired\',\n        \'stochastic_summary\':\'metric per run, then arithmetic mean; no best-run selection\',\n        \'interval_scope\':\'conditional on frozen training split, modes, mapping, and saved initialization runs\',\n        \'unresolved\':\'included as incorrect/abstention; two reference poles used for macro-F1; coverage separately reported\',\n        \'multiplicity\':\'individual paired CIs descriptive; primary seven overall mean-BA contrasts also have Bonferroni percentile family intervals\',\n        \'semantic_scope\':\'direction correspondence; behavioral 11-unit magnitudes are not questionnaire scores\'})\n    paths.append(path)\n    cache_write(files,checkpoint,signature,paths,{\'n_test\':n,\'n_methods\':8,\'bootstrap_valid\':int(valid.sum())})\n    return pd.DataFrame(overall),False\n\n\ndef stability_job(files,output,control,context,upstream,mapping,reference,code_state,cfg):\n    from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score,adjusted_mutual_info_score\n    method=upstream[\'identity\'][\'method\'];rep=context[\'rep\'];identity=upstream[\'identity\'];key=digest(identity)[:24]\n    signature=digest({\'code\':code_state,\'upstream\':upstream[\'checksum\'],\'mapping\':mapping.to_dict(\'records\'),\n        \'reference\':reference[\'signature\'],\'save_members\':cfg[\'save_member_comparisons\']})\n    path=control/\'STABILITAS\'/f\'{key}.json\';cached=cache_read(files,path,signature,cfg[\'force\'])\n    if cached: return cached[\'summary\'],\'CACHE\'\n    idx=context[\'indices\'];current=profiles(mapping,upstream[\'labels\'],context[\'ids\'],idx)\n    ref=reference[\'profiles\'].iloc[idx].reset_index(drop=True)\n    if ref.identity.tolist()!=context[\'ids\']: raise RuntimeError(\'Peserta resample tidak sejajar dengan full reference initializer.\')\n    truth=numeric_profiles(ref);pred=numeric_profiles(current);paired=(truth!=0)&(pred!=0);agreement=paired&(truth==pred)\n    complete=agreement.all(axis=1)\n    row={**identity,\'n_learners\':len(idx),\'reference_full_run\':0,\'reference_seed\':4161600,\n        \'mean_direction_agreement\':float(agreement.mean()),\'all_four_direction_agreement\':float(complete.mean()),\'base_ls_agreement\':float(complete.mean()),\n        \'reference_complete_coverage\':float((truth!=0).all(axis=1).mean()),\'resample_complete_coverage\':float((pred!=0).all(axis=1).mean()),\n        \'unresolved_learner_rate\':float((pred==0).any(axis=1).mean()),\'joint_resolved_dimensional_coverage\':float(paired.mean()),\n        \'n_resolved_profiles\':int(current.loc[current.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique()),\n        \'ari\':float(adjusted_rand_score(ref.cluster_id,current.cluster_id)),\n        \'nmi\':float(normalized_mutual_info_score(ref.cluster_id,current.cluster_id)),\n        \'ami\':float(adjusted_mutual_info_score(ref.cluster_id,current.cluster_id))}\n    for di,keydim in enumerate(KEYS):\n        row[keydim+\'_agreement\']=float(agreement[:,di].mean())\n        row[keydim+\'_joint_resolved_coverage\']=float(paired[:,di].mean())\n        row[keydim+\'_resolved_only_agreement\']=float(agreement[:,di].sum()/paired[:,di].sum()) if paired[:,di].sum() else np.nan\n    refscore=ref[[k+\'_score\' for k in KEYS]].to_numpy(float);score=current[[k+\'_score\' for k in KEYS]].to_numpy(float)\n    row[\'mean_absolute_signed_score_change\']=float(np.nanmean(np.abs(score-refscore))) if paired.any() else np.nan\n    paths=[]\n    if cfg[\'save_member_comparisons\']:\n        frame=pd.DataFrame({\'identity\':context[\'ids\'],\'source_row_index\':idx,\'reference_cluster_id\':ref.cluster_id,\n                            \'resample_cluster_id\':current.cluster_id,\'reference_ls_class\':ref.ls_class,\'resample_ls_class\':current.ls_class,\n                            \'all_four_agreement\':complete})\n        for di,k in enumerate(KEYS):\n            frame[k+\'_reference\']=ref[k];frame[k+\'_resample\']=current[k];frame[k+\'_agreement\']=agreement[:,di]\n        member_path=output/\'LMS_STABILITAS/ANGGOTA\'/(\'rep_%02d_\' % rep+re.sub(r\'[^A-Za-z0-9]+\',\'_\',method.replace(\'++\',\'_PP\'))+\'.csv.gz\')\n        save_csv(member_path,frame);paths.append(member_path)\n    cache_write(files,path,signature,paths,{\'summary\':clean(row),\'identity\':identity})\n    return row,\'CALCULATE\'\n\n\ndef summarize_stability(rows):\n    raw=pd.DataFrame(rows);summary=[]\n    metrics=[\'mean_direction_agreement\',\'all_four_direction_agreement\',\'base_ls_agreement\',\'unresolved_learner_rate\',\n             \'joint_resolved_dimensional_coverage\',\'n_resolved_profiles\',\'ari\',\'nmi\',\'ami\']+[k+\'_agreement\' for k in KEYS]\n    for method in ORDER:\n        frame=raw.loc[raw.method.eq(method)]\n        if len(frame)!=30 or frame.rep.nunique()!=30: raise RuntimeError(\'Stabilitas harus berisi 30 resample per metode.\')\n        row={\'method\':method,\'n_resamples\':30}\n        for metric in metrics:\n            row[metric+\'_mean\']=frame[metric].mean();row[metric+\'_sd\']=frame[metric].std(ddof=1)\n            row[metric+\'_min\']=frame[metric].min();row[metric+\'_max\']=frame[metric].max()\n        summary.append(row)\n    paired=[];care=raw.loc[raw.method.eq(\'CARE-KM\')].set_index(\'rep\')\n    for method in [m for m in ORDER if m!=\'CARE-KM\']:\n        other=raw.loc[raw.method.eq(method)].set_index(\'rep\').loc[care.index]\n        for metric in [\'mean_direction_agreement\',\'base_ls_agreement\',\'ari\']:\n            delta=care[metric]-other[metric]\n            paired.append({\'baseline\':method,\'metric\':metric,\'care_minus_baseline_mean\':delta.mean(),\'sd_matched_differences\':delta.std(ddof=1),\n                           \'wins\':int((delta>1e-12).sum()),\'ties\':int((delta.abs()<=1e-12).sum()),\'losses\':int((delta<-1e-12).sum()),\n                           \'interpretation\':\'descriptive matched overlapping resamples; not 30 independent cohorts\'})\n    return raw,pd.DataFrame(summary),pd.DataFrame(paired)\n\n\ndef figures(output,ils,stability):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    paths=[];folder=output/\'GAMBAR\';folder.mkdir(parents=True,exist_ok=True)\n    data=ils.set_index(\'method\').loc[ORDER];y=np.arange(len(data))\n    fig,ax=plt.subplots(figsize=(8.5,5))\n    values=data.balanced_accuracy_mean.to_numpy()*100\n    # Percentile CIs need not contain the point estimate; draw their actual endpoints.\n    low=data.balanced_accuracy_ci95_low.to_numpy()*100;high=data.balanced_accuracy_ci95_high.to_numpy()*100\n    ax.hlines(y,low,high,color=\'#333333\');ax.plot(values,y,\'o\',color=\'#185785\')\n    ax.set_yticks(y,ORDER);ax.invert_yaxis();ax.set_xlabel(\'Mean balanced accuracy (%)\');ax.set_xlim(0,100)\n    ax.set_title(\'Pendikom holdout: direct profiles and paired learner bootstrap CIs\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'01_ils_mean_balanced_accuracy.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9,5));groups=[stability.loc[stability.method.eq(m),\'mean_direction_agreement\'].to_numpy()*100 for m in ORDER]\n    ax.boxplot(groups,positions=np.arange(1,9),widths=.55)\n    ax.set_xticks(np.arange(1,9),ORDER,rotation=30,ha=\'right\');ax.set_ylim(0,100)\n    ax.set_ylabel(\'Mean agreement of four directions (%)\');ax.set_title(\'LMS semantic stability across 30 matched 80% resamples\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'02_lms_semantic_stability.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    return paths\n\n\ndef unchanged(files,watched,code_checks):\n    for path,checksum in watched.items():\n        if files.hash(path)!=checksum: raise RuntimeError(\'Masukan berubah selama eksekusi: \'+path)\n    for stem,tag,var,expected in code_checks:\n        if notebook_hash(files,stem,tag,var)!=expected: raise RuntimeError(\'Kode notebook berubah selama eksekusi: \'+stem)\n\n\ndef run_semantics(root,configuration,embedded_source):\n    root=Path(root).expanduser().resolve();files=Files(root);cfg=check_config(configuration)\n    protocol,rules,watched=input_setup(files)\n    own_hash=notebook_hash(files,STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\')\n    code_checks=[(UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\',protocol[\'code\'][\'notebook_code\']),\n                 (STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\',own_hash)]\n    code_state={\'implementation_sha256\':sha(embedded_source.encode(\'utf-8\')),\'notebook_code\':own_hash,\'runtime\':runtime()}\n    output=root/\'OUTPUT/TAMBAHAN\'/STAGE;control=output/\'KONTROL\'\n    output.mkdir(parents=True,exist_ok=True);control.mkdir(parents=True,exist_ok=True)\n    save_json(output/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code_state,\'upstream_protocol\':protocol,\n        \'semantic_rule\':\'mode-first; member tie-break; unchanged member-derived 11-unit magnitude\',\n        \'primary_metric\':\'Pendikom mean balanced accuracy\',\'stability_reference\':\'own initializer full run 0; no reference chosen by outcomes\',\n        \'method_count\':8,\'mapping_jobs_expected\':432,\'stability_jobs_expected\':240,\'bootstrap_unit\':\'learner; all methods/runs/dimensions paired\',\n        \'unknown_vs_balanced\':\'UNRESOLVED direction differs from Balanced preference magnitude {1,3}\',\n        \'no_new_clustering\':True})\n    for ds in [\'LMS\',\'PENDIKOM\']: save_csv(output/\'PROTOKOL\'/(\'02_evidence_\'+ds+\'.csv\'),rules[ds])\n    counts={\'CACHE\':0,\'MAP\':0};stability_counts={\'CACHE\':0,\'CALCULATE\':0};rows=[];artifacts=[];stable=[];references={}\n    predictions={m:[] for m in METHODS};reference=None;truth=None;all_watched=dict(watched)\n    try:\n        for ci,context in enumerate(contexts(files),1):\n            unchanged(files,watched,code_checks)\n            print(f"[{ci}/33] {context[\'scope\']} | rep={context[\'rep\']} | n={len(context[\'X\'])}",flush=True)\n            for method in METHODS:\n                runs=15 if method in STOCHASTIC and context[\'scope\']!=\'LMS_RESAMPLING\' else 1\n                for run in range(runs):\n                    upstream=upstream_job(files,protocol,context,method,run);all_watched.update(upstream[\'watched\'])\n                    mapping,evidence,record,summary,disposition=semantic_job(files,output,control,context,upstream,rules[context[\'dataset\']],code_state,cfg)\n                    counts[disposition]+=1;rows.append(summary)\n                    artifacts.append({**upstream[\'identity\'],\'upstream_record_sha256\':upstream[\'checksum\'],\n                                      \'files\':json.dumps(list(record[\'outputs\']),ensure_ascii=False),\n                                      \'upstream_files\':json.dumps(list(upstream[\'record\'][\'outputs\']),ensure_ascii=False)})\n                    if context[\'scope\']==\'LMS_FULL\' and run==0:\n                        frame=profiles(mapping,upstream[\'labels\'],context[\'ids\'],context[\'indices\'])\n                        references[method]={\'profiles\':frame,\'signature\':digest({\'upstream\':upstream[\'checksum\'],\'mapping\':mapping.to_dict(\'records\')})}\n                    if context[\'scope\']==\'LMS_RESAMPLING\':\n                        row,dispo=stability_job(files,output,control,context,upstream,mapping,references[method],code_state,cfg)\n                        stable.append(row);stability_counts[dispo]+=1\n                    if context[\'scope\']==\'PENDIKOM_TRAIN\':\n                        if reference is None: reference,truth=ils_reference(files,context)\n                        predictions[method].append(numeric_profiles(profiles(mapping,upstream[\'test_labels\'],context[\'test_ids\'])))\n            save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'completed_mapping_jobs\':len(rows),\'mapping_counts\':counts,\'last_context\':context[\'scope\']})\n        if len(rows)!=432 or len(stable)!=240: raise RuntimeError(\'Jumlah partisi/stabilitas belum lengkap.\')\n        unchanged(files,all_watched,code_checks)\n        ils,ils_cached=evaluate_ils(files,output,control,predictions,reference,truth,code_state,cfg)\n        raw,stability,paired=summarize_stability(stable)\n        save_csv(output/\'01_ringkasan_semua_partisi.csv\',pd.DataFrame(rows))\n        save_csv(output/\'02_peta_artefak.csv\',pd.DataFrame(artifacts))\n        save_csv(output/\'LMS_STABILITAS/01_metrik_per_resample.csv\',raw)\n        save_csv(output/\'LMS_STABILITAS/02_ringkasan_8_initializer.csv\',stability)\n        save_csv(output/\'LMS_STABILITAS/03_perbedaan_berpasangan_deskriptif.csv\',paired)\n        save_json(output/\'LMS_STABILITAS/04_protokol_stabilitas.json\',{\n            \'reference\':\'own-method LMS full run 0 (seed 4161600), declared in stage 25\',\n            \'resampling\':\'30 fixed matched 80% samples; overlapping learners\',\n            \'inference\':\'descriptive mean/SD/range and matched differences; no independent-cohort p-values\',\n            \'unresolved\':\'never counted as correct direction agreement; conditional agreement includes paired coverage\',\n            \'assignment_source\':\'stage 25 bank; no reclustering\', \'compressed_members\':\'CSV inside gzip; learner IDs, profiles and agreements are auditable\'})\n        summary=pd.DataFrame(rows)\n        save_csv(output/\'LMS_FULL/01_keragaman_profil_per_run.csv\',summary.loc[summary.scope.eq(\'LMS_FULL\')])\n        paper=ils.set_index(\'method\').loc[ORDER].reset_index()\n        for metric in METRIC_NAMES:\n            paper[metric+\'_mean_ci95\']=paper.apply(lambda r: f"{100*r[metric+\'_mean\']:.2f} [{100*r[metric+\'_ci95_low\']:.2f}, {100*r[metric+\'_ci95_high\']:.2f}]",axis=1)\n        save_csv(output/\'TABEL/01_initializer_ils.csv\',paper[[\'method\',\'n_runs\']+[m+\'_mean_ci95\' for m in METRIC_NAMES]+[\'coverage_mean\',\'complete_profile_coverage\']])\n        save_csv(output/\'TABEL/02_initializer_stabilitas.csv\',stability[[\'method\',\'n_resamples\',\'mean_direction_agreement_mean\',\'mean_direction_agreement_sd\',\n            \'base_ls_agreement_mean\',\'base_ls_agreement_sd\',\'unresolved_learner_rate_mean\',\'n_resolved_profiles_mean\',\'ari_mean\']])\n        # Render only when figure inputs/code changed or a figure is missing/tampered.\n        fig_signature=digest({\'code\':code_state,\'ils\':sha((output/\'ILS/04_ringkasan_overall.csv\').read_bytes()),\n                              \'stability\':sha((output/\'LMS_STABILITAS/01_metrik_per_resample.csv\').read_bytes())})\n        fig_cache=cache_read(files,control/\'gambar.json\',fig_signature,cfg[\'force\'])\n        if not fig_cache: cache_write(files,control/\'gambar.json\',fig_signature,figures(output,ils,raw),{})\n        unchanged(files,all_watched,code_checks)\n        save_json(output/\'PROTOKOL/03_manifest_masukan.json\',all_watched)\n        status={\'status\':\'selesai\',\'completed_mapping_jobs\':432,\'completed_stability_jobs\':240,\'method_count\':8,\n                \'mapping_counts\':counts,\'stability_counts\':stability_counts,\'ils_metrics_cached\':ils_cached,\n                \'full_protocol\':cfg[\'bootstrap_repetitions\']>=10000,\'n_ils_test\':139,\n                \'new_clustering_runs\':0,\'output\':str(output),\'finished_utc\':datetime.now(timezone.utc).isoformat()}\n        # Manifest includes stage-26 scientific deliverables, not mutable control logs.\n        outputs={str(p.relative_to(root)).replace(\'\\\\\',\'/\'):sha(p.read_bytes()) for p in output.rglob(\'*\') if p.is_file() and control not in p.parents}\n        save_json(control/\'manifest_keluaran.json\',outputs);save_json(control/\'status_eksekusi.json\',status)\n        print(\'Tahap 26 selesai. Partisi:\',counts,\'| Stabilitas:\',stability_counts,\'| ILS cache:\',ils_cached,flush=True)\n        print(\'Hasil:\',output)\n        return {\'ils\':ils,\'stability\':stability,\'status\':status,\'output\':str(output)}\n    except BaseException as error:\n        save_json(control/\'status_eksekusi.json\',{\'status\':\'terhenti\',\'completed_mapping_jobs\':len(rows),\n                  \'mapping_counts\':counts,\'error\':type(error).__name__+\': \'+str(error),\n                  \'resume\':\'jalankan notebook 26 kembali; cache diperiksa berdasarkan isi\'})\n        raise\n'

_s26 = types.ModuleType('_stage26_helpers')
exec(compile(UPSTREAM26_SOURCE, '_stage26_helpers', 'exec'), _s26.__dict__)
_s26.EMBEDDED_SOURCE = UPSTREAM26_SOURCE


## Evaluasi berpasangan dan interpretasi

Ukuran utama adalah mean balanced accuracy pada empat dimensi ILS,
disertai accuracy, macro-F1, confusion matrices, reference counts, dan
coverage. Seluruh 139 peserta tetap masuk denominator; unresolved dihitung
sebagai abstention/kesalahan arah. Metrik resolved-only memiliki denominator
terpisah. Prediksi seluruh metode, operator, run, dan empat dimensi
peserta tetap berpasangan pada 10.000 bootstrap peserta.

Tiga kontras utama adalah CARE-KM mode-first terhadap direct, member-only,
dan member-first. CI individual bersifat deskriptif; untuk mean BA utama
tersedia interval percentile Bonferroni keluarga tiga kontras. Ini berbeda
dari keluarga tujuh perbandingan initializer pada tahap 26. Bobot bootstrap
eksplisit disimpan agar analisis gabungan dapat memakainya kembali.

Direct bersifat invariant untuk arah peserta yang resolved ketika hanya
daftar peserta resample berubah. Sifat konstruksi ini bukan bukti keunggulan
empiris robustness. Unresolved tetap memengaruhi agreement dengan denominator
seluruh peserta. Tiga puluh resample yang tumpang tindih diringkas deskriptif.

Konflik mode–anggota adalah sifat evidence pada partisi tetap. Konflik
arah yang dilaporkan–anggota dapat berubah ketika operator diganti; kedua
hitungan dipisahkan. Hasil identik antaroperator tetap dicatat apa adanya.


In [4]:
PROFILE27_SOURCE = '"""Profile-construction controls on the frozen stage-25/26 partition bank."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport io\nimport json\nimport re\nimport numpy as np\nimport pandas as pd\n\nSTAGE=\'27_Perbandingan_Pembentukan_Profil\'\nPRIMARY=\'PRIMARY_MODE_FIRST\'\nDIRECT=\'DIRECT_BEHAVIORAL_RULES\'\nMEMBER=\'MEMBER_ONLY\'\nCONSISTENT=\'DIRECTION_CONSISTENT_MEMBER_FIRST\'\nRULES=[PRIMARY,MEMBER,CONSISTENT]\nDISPLAY_RULES=[PRIMARY,DIRECT,MEMBER,CONSISTENT]\nU26=\'OUTPUT/TAMBAHAN/26_Pengaruh_Initializer_Profil_Semantik\'\nDEFAULT_CONFIG={\'bootstrap_repetitions\':10000,\'bootstrap_seed\':20261014,\'save_member_comparisons\':True,\'force\':False}\n\n\ndef pid(initializer,rule):\n    return DIRECT if rule==DIRECT else initializer+\'|\'+rule\n\n\ndef registry(s):\n    return [{\'pathway\':pid(m,r),\'initializer\':m,\'rule\':r} for m in s.METHODS for r in RULES]+[\n        {\'pathway\':DIRECT,\'initializer\':\'NONE\',\'rule\':DIRECT}]\n\n\ndef config(s,configuration):\n    if set(configuration)-set(DEFAULT_CONFIG):raise ValueError(\'Konfigurasi tahap 27 tidak dikenal.\')\n    return s.check_config(configuration)\n\n\ndef read26(s,files):\n    protocol25,rules,watched=s.input_setup(files)\n    required=[U26+\'/PROTOKOL/01_konfigurasi.json\',U26+\'/KONTROL/status_eksekusi.json\']\n    missing=[p for p in required if not files.path(p).is_file()]\n    if missing:raise FileNotFoundError(\'Selesaikan tahap 26 terlebih dahulu: \'+str(missing))\n    status=json.loads(files.read(required[1]));p26=json.loads(files.read(required[0]))\n    if (status.get(\'status\')!=\'selesai\' or status.get(\'completed_mapping_jobs\')!=432\n        or status.get(\'completed_stability_jobs\')!=240 or status.get(\'method_count\')!=8\n        or status.get(\'n_ils_test\')!=139 or status.get(\'full_protocol\') is not True):\n        raise RuntimeError(\'Tahap 26 belum selesai dengan protokol penuh.\')\n    if p26[\'code\'][\'implementation_sha256\']!=s.sha(s.EMBEDDED_SOURCE.encode(\'utf-8\')):\n        raise RuntimeError(\'Implementasi sumber tahap 26 berbeda dari versi yang didukung notebook 27.\')\n    actual=s.notebook_hash(files,s.STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\')\n    if actual!=p26[\'code\'][\'notebook_code\']:raise RuntimeError(\'Kode notebook 26 berubah setelah hasil dibentuk.\')\n    if p26[\'upstream_protocol\']!=protocol25:raise RuntimeError(\'Protokol sumber tahap 25/26 tidak sama.\')\n    watched[required[0]]=files.hash(required[0])\n    return protocol25,p26,rules,watched\n\n\ndef primary26(s,files,p26,ctx,up,rules):\n    identity=up[\'identity\'];key=s.digest(identity)[:24]\n    path=files.path(U26+\'/KONTROL/PROFIL/\'+key+\'.json\')\n    signature=s.digest({\'code\':p26[\'code\'],\'upstream_record_sha256\':up[\'checksum\'],\n        \'matrix_sha256\':ctx[\'matrix_sha256\'],\'evidence\':rules.to_dict(\'records\'),\'rule\':\'mode-first-member-tie-11eq-v1\'})\n    record=s.cache_read(files,path,signature)\n    if record is None or record.get(\'identity\')!=identity or record.get(\'upstream_artifacts\')!=up[\'watched\']:\n        raise RuntimeError(\'Hasil profil tahap 26 berubah/tidak sesuai. Pulihkan atau jalankan kembali 26: \'+str(identity))\n    def find(filename):\n        found=[p for p in record[\'outputs\'] if p.endswith(\'/\'+filename)]\n        if len(found)!=1:raise RuntimeError(\'Artefak profil 26 tidak lengkap: \'+filename)\n        return found[0]\n    mapping=files.frame(find(\'01_cluster_profiles.csv\'));evidence=files.frame(find(\'02_cluster_dimension_evidence.csv\'))\n    if mapping.cluster_id.tolist()!=list(range(16)) or not np.array_equal(mapping.cluster_size,np.bincount(up[\'labels\'],minlength=16)):\n        raise RuntimeError(\'Profil 26 tidak sejajar dengan cluster sumber.\')\n    expected=[(i,d) for i in range(16) for d in s.DIMS]\n    if list(zip(evidence.cluster_id,evidence.dimension))!=expected:raise RuntimeError(\'Urutan evidence 26 tidak sesuai.\')\n    watched={str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'):files.hash(str(path.relative_to(files.root)).replace(\'\\\\\',\'/\')),**record[\'outputs\']}\n    return {\'mapping\':mapping,\'evidence\':evidence,\'record\':record,\'signature\':signature,\n            \'mapping_path\':find(\'01_cluster_profiles.csv\'),\'evidence_path\':find(\'02_cluster_dimension_evidence.csv\'),\n            \'watched\':watched,\'find\':find}\n\n\ndef make_mapping(s,primary,evidence,rule):\n    """Change only the direction operator; reuse frozen P/N/Z and magnitude."""\n    if rule not in [MEMBER,CONSISTENT]:raise ValueError(\'Operator tidak didukung.\')\n    mapping=primary.copy();mapping[\'reported_member_disagreements\']=0;decisions=[]\n    for cid in range(16):\n        poles_selected=[];sources=[];reported=0;mode_conflicts=0\n        for di,(dim,poles,key) in enumerate(zip(s.DIMS,s.POLES,s.KEYS)):\n            e=evidence.iloc[cid*4+di];ms=int(e.mode_score);ns=int(e.member_score)\n            selected=int(np.sign(ns)) if ns else (int(np.sign(ms)) if rule==CONSISTENT else 0)\n            source=\'MEMBER\' if ns else \'MODE\' if selected else \'BALANCED\';sources.append(source)\n            mag=int(e.preference_11eq_magnitude)\n            if mag not in {1,3,5,7,9,11}:raise RuntimeError(\'Magnitudo sumber tidak valid.\')\n            pole=poles[0] if selected==1 else poles[1] if selected==-1 else None\n            short=s.ABBR[pole] if pole else \'Balanced\';poles_selected.append(pole)\n            conflict=bool(selected and ns and selected!=np.sign(ns));underlying=bool(ms*ns<0)\n            reported+=conflict;mode_conflicts+=underlying\n            mapping.loc[cid,key]=short;mapping.loc[cid,key+\'_score\']=selected*mag if selected else np.nan\n            mapping.loc[cid,key+\'_resolved\']=bool(selected);mapping.loc[cid,key+\'_code\']=short+str(mag) if selected else \'Balanced\'\n            decisions.append({\'cluster_id\':cid,\'dimension\':dim,\'cluster_size\':int(e.cluster_size),\'mode_score\':ms,\'member_score\':ns,\n                \'selected_pole\':short,\'decision_source\':source,\'magnitude\':mag,\'preference_category\':e.preference_category,\n                \'mode_member_conflict\':underlying,\'reported_member_conflict\':conflict,\n                \'strong_reported_member_conflict\':conflict and mag in (9,11),\'rule\':rule})\n        resolved=all(p is not None for p in poles_selected)\n        mapping.loc[cid,\'ls_class\']=s.LS_LOOKUP[tuple(poles_selected)] if resolved else \'AMBIGUOUS\'\n        mapping.loc[cid,\'semantic_label\']=\'-\'.join(mapping.loc[cid,k] for k in s.KEYS)\n        mapping.loc[cid,\'preference_signature\']=\'|\'.join(mapping.loc[cid,k+\'_code\'] for k in s.KEYS)\n        mapping.loc[cid,\'mapping_status\']=\'RESOLVED\' if resolved else \'UNRESOLVED\'\n        mapping.loc[cid,\'mode_dimensions\']=sources.count(\'MODE\');mapping.loc[cid,\'member_dimensions\']=sources.count(\'MEMBER\')\n        mapping.loc[cid,\'balanced_dimensions\']=sources.count(\'BALANCED\');mapping.loc[cid,\'mode_member_disagreements\']=mode_conflicts\n        mapping.loc[cid,\'reported_member_disagreements\']=reported\n    return mapping,pd.DataFrame(decisions)\n\n\ndef primary_decisions(s,evidence):\n    rows=[]\n    for e in evidence.itertuples(index=False):\n        di=s.DIMS.index(e.dimension);poles=s.POLES[di]\n        selected=1 if e.selected_pole==s.ABBR[poles[0]] else -1 if e.selected_pole==s.ABBR[poles[1]] else 0\n        rows.append({\'cluster_id\':int(e.cluster_id),\'dimension\':e.dimension,\'cluster_size\':int(e.cluster_size),\n            \'mode_score\':int(e.mode_score),\'member_score\':int(e.member_score),\'selected_pole\':e.selected_pole,\n            \'decision_source\':e.decision_source,\'magnitude\':int(e.preference_11eq_magnitude),\'preference_category\':e.preference_category,\n            \'mode_member_conflict\':bool(e.mode_score*e.member_score<0),\n            \'reported_member_conflict\':bool(selected and e.member_score and selected!=np.sign(e.member_score)),\n            \'strong_reported_member_conflict\':bool(selected and e.member_score and selected!=np.sign(e.member_score) and e.preference_11eq_magnitude in (9,11)),\n            \'rule\':PRIMARY})\n    return pd.DataFrame(rows)\n\n\ndef summarize_mapping(s,ctx,up,mapping,decisions,rule,mapping_path):\n    n=len(ctx[\'X\']);sizes=mapping.cluster_size.to_numpy();resolved=mapping.mapping_status.eq(\'RESOLVED\')\n    mass=mapping.loc[resolved].groupby(\'ls_class\').cluster_size.sum()\n    return {**up[\'identity\'],\'initializer\':up[\'identity\'][\'method\'],\'rule\':rule,\'pathway\':pid(up[\'identity\'][\'method\'],rule),\n        \'n_learners\':n,\'n_clusters\':16,\'n_resolved_profiles\':int(mass.size),\'unresolved_learner_rate\':float(sizes[~resolved].sum()/n),\n        \'largest_resolved_profile_share_all_learners\':float(mass.max()/n) if len(mass) else 0.,\n        \'n_cluster_dimensions\':64,\'n_strong_decisions\':int(decisions.magnitude.isin([9,11]).sum()),\n        \'mode_member_conflicts\':int(decisions.mode_member_conflict.sum()),\'reported_member_conflicts\':int(decisions.reported_member_conflict.sum()),\n        \'strong_mode_member_conflicts\':int((decisions.mode_member_conflict&decisions.magnitude.isin([9,11])).sum()),\n        \'strong_reported_member_conflicts\':int(decisions.strong_reported_member_conflict.sum()),\n        \'weighted_mode_member_conflict_rate\':float(decisions.loc[decisions.mode_member_conflict,\'cluster_size\'].sum()/(n*4)),\n        \'weighted_reported_member_conflict_rate\':float(decisions.loc[decisions.reported_member_conflict,\'cluster_size\'].sum()/(n*4)),\n        \'mapping_file\':mapping_path}\n\n\ndef cluster_job(s,files,out,control,ctx,up,primary,rule,code,cfg):\n    identity={**up[\'identity\'],\'rule\':rule};key=s.digest(identity)[:24]\n    sig=s.digest({\'code\':code,\'source26_signature\':primary[\'signature\'],\'source26_outputs\':primary[\'record\'][\'outputs\'],\'identity\':identity})\n    checkpoint=control/\'PROFIL\'/f\'{key}.json\'\n    record=s.cache_read(files,checkpoint,sig,cfg[\'force\'])\n    folder_rule=\'MEMBER_FIRST\' if rule==CONSISTENT else rule\n    folder=out/\'BANK\'/folder_rule/ctx[\'scope\']/ctx[\'dataset\']/(\'r%02d\'%(ctx[\'rep\'] if ctx[\'rep\'] is not None else up[\'identity\'][\'run\']))/re.sub(r\'[^A-Za-z0-9]+\',\'_\',up[\'identity\'][\'method\'].replace(\'++\',\'_PP\'))\n    mp=folder/\'01_cluster_profiles.csv\';ep=folder/\'02_direction_decisions.csv\'\n    if record:\n        mapping=files.frame(str(mp.relative_to(files.root)).replace(\'\\\\\',\'/\'));decisions=files.frame(str(ep.relative_to(files.root)).replace(\'\\\\\',\'/\'));dispo=\'CACHE\'\n    else:\n        mapping,decisions=make_mapping(s,primary[\'mapping\'],primary[\'evidence\'],rule)\n        s.save_csv(mp,mapping);s.save_csv(ep,decisions);paths=[mp,ep]\n        if ctx[\'test_X\'] is not None:\n            for name,frame in [(\'03_train_targets.csv.gz\',s.profiles(mapping,up[\'labels\'],ctx[\'ids\'],ctx[\'indices\'])),\n                (\'04_test_targets.csv.gz\',s.profiles(mapping,up[\'test_labels\'],ctx[\'test_ids\']))]:\n                path=folder/name;s.save_csv(path,frame);paths.append(path)\n        record=s.cache_write(files,checkpoint,sig,paths,{\'identity\':identity,\'source26_artifacts\':primary[\'record\'][\'outputs\']});dispo=\'MAP\'\n    return mapping,decisions,record,summarize_mapping(s,ctx,up,mapping,decisions,rule,str(mp.relative_to(files.root)).replace(\'\\\\\',\'/\')),dispo\n\n\ndef direct_profiles(s,X,ids,features,rules,indices=None):\n    n=len(X);frame=pd.DataFrame({\'identity\':ids,\'cluster_id\':[None]*n});evidence=[];codes=[]\n    if indices is not None:frame.insert(1,\'source_row_index\',indices)\n    fpos={f:i for i,f in enumerate(features)}\n    for dim,key,poles in zip(s.DIMS,s.KEYS,s.POLES):\n        rr=rules.loc[rules.dimension.eq(dim)];cols=np.array([fpos[f] for f in rr.feature]);lo=np.array([1 if p==poles[0] else -1 for p in rr.low_supports]);hi=np.array([1 if p==poles[0] else -1 for p in rr.high_supports])\n        data=X[:,cols];vote=np.where(data==\'1\',lo,np.where(data==\'3\',hi,0))\n        pos=(vote==1).sum(axis=1);neg=(vote==-1).sum(axis=1);neu=(vote==0).sum(axis=1);balance=pos-neg;direction=np.sign(balance).astype(int);codes.append(direction)\n        units=np.floor(11*(pos+.5*neu)/(pos+neg+neu)+.5).astype(int);mag=np.abs(2*units-11)\n        short=np.where(direction==1,s.ABBR[poles[0]],np.where(direction==-1,s.ABBR[poles[1]],\'Balanced\'))\n        frame[key]=short;frame[key+\'_score\']=np.where(direction!=0,direction*mag,np.nan)\n        frame[key+\'_category\']=np.where(np.isin(mag,[1,3]),\'Balanced\',np.where(np.isin(mag,[5,7]),\'Moderate preference\',\'Strong preference\'))\n        frame[key+\'_code\']=np.where(direction!=0,np.char.add(short.astype(str),mag.astype(str)),\'Balanced\')\n        evidence.append(pd.DataFrame({\'identity\':ids,\'dimension\':dim,\'positive\':pos,\'negative\':neg,\'neutral\':neu,\'balance\':balance,\n            \'selected_pole\':short,\'magnitude\':mag,\'resolved\':direction!=0,\'rule\':DIRECT}))\n    full=np.column_stack(codes);resolved=(full!=0).all(axis=1)\n    frame[\'mapping_status\']=np.where(resolved,\'RESOLVED\',\'UNRESOLVED\')\n    frame[\'ls_class\']=[s.LS_LOOKUP[tuple(poles[0] if value==1 else poles[1] for value,poles in zip(row,s.POLES))] if ok else \'AMBIGUOUS\' for row,ok in zip(full,resolved)]\n    return frame,pd.concat(evidence,ignore_index=True)\n\n\ndef direct_job(s,files,out,control,ctx,rules,code,cfg):\n    identity={\'scope\':ctx[\'scope\'],\'dataset\':ctx[\'dataset\'],\'rule\':DIRECT};sig=s.digest({\'identity\':identity,\'code\':code,\'matrix\':ctx[\'matrix_sha256\'],\'rules\':rules.to_dict(\'records\')})\n    path=control/\'DIRECT\'/(s.digest(identity)[:24]+\'.json\');record=s.cache_read(files,path,sig,cfg[\'force\'])\n    folder=out/\'DIRECT\'/ctx[\'scope\']/ctx[\'dataset\'];tp=folder/\'03_train_targets.csv.gz\';ep=folder/\'04_test_targets.csv.gz\'\n    if record:\n        train=files.frame(str(tp.relative_to(files.root)).replace(\'\\\\\',\'/\'),True)\n        for k in s.KEYS:train[k+\'_score\']=pd.to_numeric(train[k+\'_score\'],errors=\'coerce\')\n        test=files.frame(str(ep.relative_to(files.root)).replace(\'\\\\\',\'/\'),True) if ctx[\'test_X\'] is not None else None;dispo=\'CACHE\'\n    else:\n        train,e=direct_profiles(s,ctx[\'X\'],ctx[\'ids\'],ctx[\'features\'],rules,ctx[\'indices\']);s.save_csv(tp,train)\n        p=folder/\'01_train_evidence.csv.gz\';s.save_csv(p,e);paths=[tp,p];test=None\n        if ctx[\'test_X\'] is not None:\n            test,e=direct_profiles(s,ctx[\'test_X\'],ctx[\'test_ids\'],ctx[\'features\'],rules)\n            s.save_csv(ep,test);p=folder/\'02_test_evidence.csv.gz\';s.save_csv(p,e);paths.extend([ep,p])\n        record=s.cache_write(files,path,sig,paths,{\'identity\':identity});dispo=\'DIRECT\'\n    resolved=train.mapping_status.eq(\'RESOLVED\');distribution=train.loc[resolved,\'ls_class\'].value_counts()\n    summary={**identity,\'initializer\':\'NONE\',\'method\':\'NONE\',\'pathway\':DIRECT,\'run\':0,\'rep\':None,\'n_learners\':len(train),\'n_clusters\':None,\n        \'n_resolved_profiles\':len(distribution),\'unresolved_learner_rate\':float((~resolved).mean()),\n        \'largest_resolved_profile_share_all_learners\':float(distribution.max()/len(train)) if len(distribution) else 0.,\n        \'mapping_file\':str(tp.relative_to(files.root)).replace(\'\\\\\',\'/\')}\n    return train,test,record,summary,dispo\n\n\ndef primary_stability(s,files,p26,ctx,up,primary,reference):\n    path=files.path(U26+\'/KONTROL/STABILITAS/\'+s.digest(up[\'identity\'])[:24]+\'.json\')\n    signature=s.digest({\'code\':p26[\'code\'],\'upstream\':up[\'checksum\'],\'mapping\':primary[\'mapping\'].to_dict(\'records\'),\n        \'reference\':reference[\'signature\'],\'save_members\':p26[\'configuration\'][\'save_member_comparisons\']})\n    record=s.cache_read(files,path,signature)\n    if record is None:raise RuntimeError(\'Artefak stabilitas primary tahap 26 berubah/tidak sesuai: \'+str(up[\'identity\']))\n    watched={str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'):files.hash(str(path.relative_to(files.root)).replace(\'\\\\\',\'/\')),**record[\'outputs\']}\n    return record[\'summary\'],watched\n\n\ndef direct_stability(s,files,out,control,ctx,full,code,cfg):\n    identity={\'scope\':\'LMS_RESAMPLING\',\'rule\':DIRECT,\'rep\':ctx[\'rep\']};signature=s.digest({\'identity\':identity,\'code\':code,\'full_signature\':full[\'signature\'],\'matrix\':ctx[\'matrix_sha256\'],\'save_members\':cfg[\'save_member_comparisons\']})\n    checkpoint=control/\'DIRECT_STABILITAS\'/(s.digest(identity)[:24]+\'.json\');cached=s.cache_read(files,checkpoint,signature,cfg[\'force\'])\n    if cached:return cached[\'summary\'],\'CACHE\'\n    frame=full[\'profiles\'].iloc[ctx[\'indices\']].reset_index(drop=True)\n    if frame.identity.tolist()!=ctx[\'ids\']:raise RuntimeError(\'Direct reference tidak sejajar dengan identitas resample.\')\n    codes=s.numeric_profiles(frame);resolved=codes!=0;complete=resolved.all(axis=1)\n    row={\'method\':DIRECT,\'pathway\':DIRECT,\'initializer\':\'NONE\',\'rule\':DIRECT,\'rep\':ctx[\'rep\'],\'n_learners\':len(frame),\n        \'mean_direction_agreement\':float(resolved.mean()),\'all_four_direction_agreement\':float(complete.mean()),\'base_ls_agreement\':float(complete.mean()),\n        \'unresolved_learner_rate\':float((~complete).mean()),\'joint_resolved_dimensional_coverage\':float(resolved.mean()),\n        \'resolved_only_direction_agreement\':1.0 if resolved.any() else np.nan,\'structurally_invariant_on_resolved\':True,\n        \'n_resolved_profiles\':int(frame.loc[complete,\'ls_class\'].nunique()),\'ari\':None,\'nmi\':None,\'ami\':None}\n    for di,k in enumerate(s.KEYS):row[k+\'_agreement\']=float(resolved[:,di].mean())\n    paths=[]\n    if cfg[\'save_member_comparisons\']:\n        member=frame[[\'identity\',\'ls_class\']+s.KEYS].copy();member.insert(1,\'source_row_index\',ctx[\'indices\'])\n        member[\'all_four_agreement\']=complete\n        for di,k in enumerate(s.KEYS):member[k+\'_agreement\']=resolved[:,di]\n        path=out/\'LMS_STABILITAS/DIRECT_ANGGOTA\'/(\'rep_%02d.csv.gz\'%ctx[\'rep\']);s.save_csv(path,member);paths.append(path)\n    s.cache_write(files,checkpoint,signature,paths,{\'identity\':identity,\'summary\':row})\n    return row,\'CALCULATE\'\n\n\ndef alternative_stability(s,files,out,control,ctx,up,mapping,reference,partition_metrics,rule,code,cfg):\n    method=up[\'identity\'][\'method\'];pathid=pid(method,rule);identity={**up[\'identity\'],\'rule\':rule}\n    signature=s.digest({\'code\':code,\'identity\':identity,\'upstream\':up[\'checksum\'],\'mapping\':mapping.to_dict(\'records\'),\n        \'reference\':reference[\'signature\'],\'partition_metrics\':{k:partition_metrics[k] for k in [\'ari\',\'nmi\',\'ami\']},\n        \'save_members\':cfg[\'save_member_comparisons\']})\n    checkpoint=control/\'STABILITAS\'/(s.digest(identity)[:24]+\'.json\');cached=s.cache_read(files,checkpoint,signature,cfg[\'force\'])\n    if cached:return cached[\'summary\'],\'CACHE\'\n    current=s.profiles(mapping,up[\'labels\'],ctx[\'ids\'],ctx[\'indices\']);ref=reference[\'profiles\'].iloc[ctx[\'indices\']].reset_index(drop=True)\n    if ref.identity.tolist()!=ctx[\'ids\']:raise RuntimeError(\'Identitas stabilitas alternatif tidak sejajar.\')\n    truth=s.numeric_profiles(ref);pred=s.numeric_profiles(current);joint=(truth!=0)&(pred!=0);agree=joint&(truth==pred);complete=agree.all(axis=1)\n    row={**up[\'identity\'],\'method\':pathid,\'pathway\':pathid,\'initializer\':method,\'rule\':rule,\'n_learners\':len(current),\n        \'reference_full_run\':0,\'reference_seed\':4161600,\'mean_direction_agreement\':float(agree.mean()),\n        \'all_four_direction_agreement\':float(complete.mean()),\'base_ls_agreement\':float(complete.mean()),\n        \'unresolved_learner_rate\':float((pred==0).any(axis=1).mean()),\'joint_resolved_dimensional_coverage\':float(joint.mean()),\n        \'n_resolved_profiles\':int(current.loc[current.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique()),\n        **{k:partition_metrics[k] for k in [\'ari\',\'nmi\',\'ami\']},\'partition_metrics_origin\':\'unchanged partition; reused stage 26\'}\n    for di,k in enumerate(s.KEYS):\n        row[k+\'_agreement\']=float(agree[:,di].mean());row[k+\'_joint_resolved_coverage\']=float(joint[:,di].mean())\n        row[k+\'_resolved_only_agreement\']=float(agree[:,di].sum()/joint[:,di].sum()) if joint[:,di].sum() else np.nan\n    refscore=ref[[k+\'_score\' for k in s.KEYS]].to_numpy(float);score=current[[k+\'_score\' for k in s.KEYS]].to_numpy(float)\n    row[\'mean_absolute_signed_score_change\']=float(np.nanmean(np.abs(score-refscore))) if joint.any() else np.nan\n    paths=[]\n    if cfg[\'save_member_comparisons\']:\n        frame=pd.DataFrame({\'identity\':ctx[\'ids\'],\'source_row_index\':ctx[\'indices\'],\'reference_cluster_id\':ref.cluster_id,\n            \'resample_cluster_id\':current.cluster_id,\'reference_ls_class\':ref.ls_class,\'resample_ls_class\':current.ls_class,\'all_four_agreement\':complete})\n        for di,k in enumerate(s.KEYS):frame[k+\'_reference\']=ref[k];frame[k+\'_resample\']=current[k];frame[k+\'_agreement\']=agree[:,di]\n        path=out/\'LMS_STABILITAS/ANGGOTA\'/(\'rep_%02d_\'%ctx[\'rep\']+re.sub(r\'[^A-Za-z0-9]+\',\'_\',pathid.replace(\'++\',\'_PP\'))+\'.csv.gz\')\n        s.save_csv(path,frame);paths.append(path)\n    s.cache_write(files,checkpoint,signature,paths,{\'identity\':identity,\'summary\':s.clean(row)})\n    return row,\'CALCULATE\'\n\n\ndef evaluate(s,files,out,control,predictions,reference,truth,code,cfg):\n    entries=registry(s);ids=[r[\'pathway\'] for r in entries];lookup={r[\'pathway\']:r for r in entries}\n    sig=s.digest({\'code\':code,\'truth\':truth.tolist(),\'ids\':reference.userid.tolist(),\n        \'predictions\':{p:s.sha(np.asarray(predictions[p],dtype=np.int8).tobytes()) for p in ids},\n        \'runs\':{p:len(predictions[p]) for p in ids},\'n_boot\':cfg[\'bootstrap_repetitions\'],\'seed\':cfg[\'bootstrap_seed\']})\n    checkpoint=control/\'evaluasi_ils.json\';folder=out/\'ILS\';cached=s.cache_read(files,checkpoint,sig,cfg[\'force\'])\n    if cached:\n        print(\'ILS dan paired CI: gunakan hasil dengan isi terverifikasi.\',flush=True)\n        return files.frame(str((folder/\'04_ringkasan_overall.csv\').relative_to(files.root)).replace(\'\\\\\',\'/\')),True\n    from threadpoolctl import threadpool_limits\n    n=len(truth);nr=cfg[\'bootstrap_repetitions\'];rng=np.random.default_rng(cfg[\'bootstrap_seed\'])\n    masks={p:s.confusion_masks(truth,np.asarray(predictions[p],dtype=np.int8)) for p in ids}\n    counts={p:masks[p].sum(axis=0) for p in ids};points={p:s.count_metrics(counts[p]) for p in ids}\n    boot=np.empty((nr,len(ids),5,3));valid=np.zeros(nr,bool);weights_all=np.empty((nr,n),np.uint16)\n    print(\'ILS:\',len(ids),\'jalur unik,\',nr,\'bootstrap peserta berpasangan.\',flush=True)\n    with threadpool_limits(limits=1):\n        for start in range(0,nr,250):\n            stop=min(start+250,nr);weights=rng.multinomial(n,np.full(n,1/n),size=stop-start)\n            weights_all[start:stop]=weights;weights=weights.astype(float)\n            pos=weights@(truth==1);valid[start:stop]=((pos>0)&(pos<n)).all(axis=1)\n            for pi,p in enumerate(ids):\n                a=masks[p];shape=a.shape;cc=(weights@a.reshape(n,-1)).reshape(stop-start,shape[1],4,6)\n                metrics=s.count_metrics(cc).mean(axis=1);boot[start:stop,pi,:4]=metrics;boot[start:stop,pi,4]=metrics.mean(axis=1)\n    if valid.sum()<100:raise RuntimeError(\'Bootstrap valid kurang dari 100.\')\n    boot=boot[valid];raw=[];cm=[];perdim=[];overall=[];pred_rows=[];refcounts=[]\n    actual=np.stack([(truth==1).sum(axis=0),(truth==-1).sum(axis=0)],axis=-1)\n    for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n        for j,pole in enumerate(poles):refcounts.append({\'dimension\':key,\'reference_pole\':pole,\'n\':int(actual[di,j]),\'n_test\':n})\n    for pi,p in enumerate(ids):\n        meta=lookup[p];arr=np.asarray(predictions[p],dtype=np.int8);values=points[p];nrun=len(arr)\n        for run in range(nrun):\n            for li,identity in enumerate(reference.userid):\n                row={**meta,\'run\':run,\'identity\':identity,\'courseid\':reference.courseid.iloc[li],\'reference_ls_class\':reference.ils_ls_class.iloc[li]};full=[]\n                for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n                    row[\'reference_\'+key]=poles[0] if truth[li,di]==1 else poles[1]\n                    v=arr[run,li,di];pole=poles[0] if v==1 else poles[1] if v==-1 else None\n                    row[\'predicted_\'+key]=pole or \'UNRESOLVED\';full.append(pole)\n                row[\'predicted_ls_class\']=s.LS_LOOKUP[tuple(full)] if all(full) else \'AMBIGUOUS\';pred_rows.append(row)\n            for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):\n                cc=counts[p][run,di];a,b,c,d,e,f=cc;resolved=n-c-f\n                raw.append({**meta,\'run\':run,\'dimension\':key,\'n_test\':n,\'reference_first\':int(actual[di,0]),\'reference_second\':int(actual[di,1]),\n                    \'n_unresolved\':int(c+f),\'n_resolved\':int(resolved),\'coverage\':resolved/n,\n                    \'resolved_only_accuracy\':(a+e)/resolved if resolved else np.nan,**dict(zip(s.METRIC_NAMES,values[run,di]))})\n                for ci,num in enumerate(cc):cm.append({**meta,\'run\':run,\'dimension\':key,\'reference_pole\':poles[ci//3],\n                    \'predicted_pole\':(poles+(\'UNRESOLVED\',))[ci%3],\'n\':int(num)})\n        for di,key in enumerate(s.KEYS+[\'mean_four_dimensions\']):\n            vv=values[:,di] if di<4 else values.mean(axis=1)\n            row={**meta,\'dimension\':key,\'n_runs\':nrun,\'n_test\':n,\'bootstrap_requested\':nr,\'bootstrap_valid\':int(valid.sum()),\n                \'coverage_mean\':float((arr[:,:,di]!=0).mean()) if di<4 else float((arr!=0).mean()),\n                \'complete_profile_coverage\':float((arr!=0).all(axis=2).mean()),\'is_main_table\':p in [pid(\'CARE-KM\',r) for r in RULES]+[DIRECT]}\n            for ji,metric in enumerate(s.METRIC_NAMES):\n                low,high=np.quantile(boot[:,pi,di,ji],[.025,.975]);row[metric+\'_mean\']=vv[:,ji].mean()\n                row[metric+\'_sd_runs\']=vv[:,ji].std(ddof=1) if nrun>1 else np.nan\n                row[metric+\'_ci95_low\']=low;row[metric+\'_ci95_high\']=high\n            (perdim if di<4 else overall).append(row)\n    contrasts=[]\n    for method in s.ORDER:\n        a=pid(method,PRIMARY);ai=ids.index(a)\n        for b in [DIRECT,pid(method,MEMBER),pid(method,CONSISTENT)]:\n            bi=ids.index(b)\n            for di,key in enumerate(s.KEYS+[\'mean_four_dimensions\']):\n                for ji,metric in enumerate(s.METRIC_NAMES):\n                    delta=boot[:,ai,di,ji]-boot[:,bi,di,ji];lo,hi=np.quantile(delta,[.025,.975])\n                    av=points[a][:,di,ji].mean() if di<4 else points[a][:,:,ji].mean()\n                    bv=points[b][:,di,ji].mean() if di<4 else points[b][:,:,ji].mean()\n                    row={\'initializer\':method,\'reference_pathway\':a,\'control_pathway\':b,\'control_rule\':lookup[b][\'rule\'],\n                        \'dimension\':key,\'metric\':metric,\'primary_minus_control\':av-bv,\'paired_ci95_low\':lo,\'paired_ci95_high\':hi,\n                        \'is_primary_family\':method==\'CARE-KM\' and di==4 and metric==\'balanced_accuracy\',\n                        \'interpretation\':\'positive favors primary mode-first; same learners and frozen partition\',\n                        \'n_learners\':n}\n                    if row[\'is_primary_family\']:\n                        row[\'bonferroni_family95_low\'],row[\'bonferroni_family95_high\']=np.quantile(delta,[.05/(2*3),1-.05/(2*3)])\n                        row[\'family\']=\'three CARE-KM overall mean-BA contrasts\'\n                    contrasts.append(row)\n    paths=[]\n    for name,frame in [(\'01_prediksi_per_peserta.csv.gz\',pd.DataFrame(pred_rows)),(\'02_metrik_per_dimensi_per_run.csv\',pd.DataFrame(raw)),\n        (\'03_ringkasan_per_dimensi.csv\',pd.DataFrame(perdim)),(\'04_ringkasan_overall.csv\',pd.DataFrame(overall)),\n        (\'05_confusion_matrices.csv\',pd.DataFrame(cm)),(\'07_perbedaan_berpasangan.csv\',pd.DataFrame(contrasts)),(\'08_reference_counts.csv\',pd.DataFrame(refcounts))]:\n        path=folder/name;s.save_csv(path,frame);paths.append(path)\n    buffer=io.BytesIO();np.savez_compressed(buffer,metrics=boot,pathways=np.asarray(ids),dimensions=np.asarray(s.KEYS+[\'mean_four_dimensions\']),\n        metric_names=np.asarray(s.METRIC_NAMES),valid_draws=valid,seed=cfg[\'bootstrap_seed\'])\n    path=folder/\'06_bootstrap_paired_metrics.npz\';s.atomic(path,buffer.getvalue());paths.append(path)\n    buffer=io.BytesIO();np.savez_compressed(buffer,weights=weights_all,valid_draws=valid,learner_ids=reference.userid.to_numpy(str),seed=cfg[\'bootstrap_seed\'])\n    path=folder/\'09_bootstrap_weights.npz\';s.atomic(path,buffer.getvalue());paths.append(path)\n    path=folder/\'10_protokol_evaluasi.json\';s.save_json(path,{\'primary_metric\':\'mean balanced accuracy across four dimensions\',\n        \'main_comparison\':\'fixed CARE-KM; primary vs direct, member-only, direction-consistent member-first\',\n        \'n_unique_pathways\':25,\'unique_direct_baseline\':1,\'n_learners\':139,\'n_valid_draws\':int(valid.sum()),\n        \'bootstrap_unit\':\'learner; all four dimensions, rules, initializers and runs paired\',\n        \'inference_scope\':\'conditional on frozen training split, modes and saved initialization runs\',\n        \'stochastic_summary\':\'all 15 runs; calculate each metric per run before averaging; direct is evaluated once\',\n        \'unresolved\':\'abstention included as error; two reference poles define macro-F1; resolved-only accuracy has separate denominator\',\n        \'multiplicity\':\'individual CIs descriptive; Bonferroni family95 only for three predeclared CARE-KM overall mean-BA contrasts\',\n        \'draws\':\'explicit learner weights saved; stage-26 primary predictions reused, evaluated jointly with controls\',\n        \'other_contrasts\':\'within-initializer rule effects and primary-vs-direct controls, supplementary/descriptive\'})\n    paths.append(path);s.cache_write(files,checkpoint,sig,paths,{\'n_pathways\':25,\'n_learners\':139})\n    return pd.DataFrame(overall),False\n\n\ndef aggregate_stability(s,rows):\n    raw=pd.DataFrame(rows);summary=[];contrasts=[]\n    metrics=[\'mean_direction_agreement\',\'base_ls_agreement\',\'unresolved_learner_rate\',\'joint_resolved_dimensional_coverage\',\'n_resolved_profiles\']\n    for p,group in raw.groupby(\'pathway\',sort=False):\n        if len(group)!=30 or group.rep.nunique()!=30:raise RuntimeError(\'Diperlukan 30 matched resample per jalur.\')\n        meta=group.iloc[0];row={k:meta[k] for k in [\'pathway\',\'initializer\',\'rule\']};row[\'n_resamples\']=30\n        row[\'structurally_invariant_on_resolved\']=p==DIRECT\n        for metric in metrics:\n            row[metric+\'_mean\']=group[metric].mean();row[metric+\'_sd\']=group[metric].std(ddof=1)\n            row[metric+\'_min\']=group[metric].min();row[metric+\'_max\']=group[metric].max()\n        summary.append(row)\n    for method in s.ORDER:\n        a=raw.loc[raw.pathway.eq(pid(method,PRIMARY))].set_index(\'rep\')\n        for rule in [MEMBER,CONSISTENT]:\n            b=raw.loc[raw.pathway.eq(pid(method,rule))].set_index(\'rep\').loc[a.index]\n            for metric in [\'mean_direction_agreement\',\'base_ls_agreement\']:\n                delta=a[metric]-b[metric];contrasts.append({\'initializer\':method,\'control_rule\':rule,\'metric\':metric,\n                    \'primary_minus_control_mean\':delta.mean(),\'sd_matched_differences\':delta.std(ddof=1),\n                    \'wins\':int((delta>1e-12).sum()),\'ties\':int((delta.abs()<=1e-12).sum()),\'losses\':int((delta<-1e-12).sum()),\n                    \'interpretation\':\'descriptive overlapping resamples; direct invariance is structural, not estimated robustness\'})\n    return raw,pd.DataFrame(summary),pd.DataFrame(contrasts)\n\n\ndef figures(s,out,ils,stability):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    names=[pid(\'CARE-KM\',r) if r!=DIRECT else DIRECT for r in DISPLAY_RULES]\n    labels=[\'Mode-first\',\'Direct rules\',\'Member-only\',\'Member-first\']\n    frame=ils.set_index(\'pathway\').loc[names];y=np.arange(4);fig,ax=plt.subplots(figsize=(8,4))\n    ax.hlines(y,100*frame.balanced_accuracy_ci95_low,100*frame.balanced_accuracy_ci95_high,color=\'#333333\')\n    ax.plot(100*frame.balanced_accuracy_mean,y,\'o\',color=\'#185785\');ax.set_yticks(y,labels);ax.invert_yaxis();ax.set_xlim(0,100)\n    ax.set_xlabel(\'Mean balanced accuracy (%)\');ax.set_title(\'Profile construction on the same Pendikom holdout\');fig.tight_layout()\n    folder=out/\'GAMBAR\';folder.mkdir(parents=True,exist_ok=True);paths=[]\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'01_pembentukan_profil_ils.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    fig,ax=plt.subplots(figsize=(8,4.5));groups=[100*stability.loc[stability.pathway.eq(p),\'mean_direction_agreement\'].to_numpy() for p in names]\n    ax.boxplot(groups,positions=np.arange(1,5));ax.set_xticks(np.arange(1,5),labels);ax.set_ylim(0,100)\n    ax.set_ylabel(\'Four-direction agreement (%)\');ax.set_title(\'LMS profiles across 30 matched 80% resamples\')\n    ax.text(.02,.02,\'Direct rules: resolved directions are fixed by construction.\\nUnresolved directions remain in the denominator.\',transform=ax.transAxes,fontsize=11)\n    fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'02_pembentukan_profil_stabilitas.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig);return paths\n\n\ndef run_profiles(root,configuration,embedded_source,s):\n    root=Path(root).expanduser().resolve();files=s.Files(root);cfg=config(s,configuration)\n    p25,p26,rules,watched=read26(s,files)\n    own=s.notebook_hash(files,STAGE,\'profile27_configuration\',\'KONFIGURASI_27\')\n    checks=[(s.UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\',p25[\'code\'][\'notebook_code\']),\n            (s.STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\',p26[\'code\'][\'notebook_code\']),\n            (STAGE,\'profile27_configuration\',\'KONFIGURASI_27\',own)]\n    code={\'implementation_sha256\':s.sha(embedded_source.encode(\'utf-8\')),\'helper26_sha256\':s.sha(s.EMBEDDED_SOURCE.encode(\'utf-8\')),\n          \'notebook_code\':own,\'runtime\':s.runtime()}\n    out=root/\'OUTPUT/TAMBAHAN\'/STAGE;control=out/\'KONTROL\';out.mkdir(parents=True,exist_ok=True);control.mkdir(parents=True,exist_ok=True)\n    s.save_json(out/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code,\'source26_code\':p26[\'code\'],\n        \'main_design\':\'fixed CARE-KM; three cluster-to-profile operators plus one direct behavioral-rule control\',\n        \'supplementary_design\':\'8 initializers x 3 operators; direct evaluated once\',\n        \'rules\':RULES+[DIRECT],\'primary_metric\':\'mean balanced accuracy on the same 139 Pendikom test learners\',\n        \'direction_consistent\':\'existing MEMBER_FIRST operator: member balance first, mode only breaks member ties; balanced magnitude 1 on member ties\',\n        \'direct_rule\':\'sum existing signed indicator evidence per learner; zero is unresolved; no learned weights or ILS calibration\',\n        \'magnitude\':\'unchanged member-derived 11-unit scale; direct projects individual indicator counts; not questionnaire scores\',\n        \'new_clustering_runs\':0,\'expected_cluster_profiles\':1296,\'expected_new_mappings\':864,\'expected_direct_contexts\':3,\'expected_stability_rows\':750})\n    for ds in [\'LMS\',\'PENDIKOM\']:s.save_csv(out/\'PROTOKOL\'/(\'02_evidence_\'+ds+\'.csv\'),rules[ds])\n    counts={\'REUSED_26\':0,\'MAP\':0,\'CACHE\':0};dcounts={\'DIRECT\':0,\'CACHE\':0};scounts={\'REUSED_26\':0,\'CALCULATE\':0,\'CACHE\':0}\n    refs={};directfull=None;rows=[];artifacts=[];stabilities=[];diagnostics=[];predictions={e[\'pathway\']:[] for e in registry(s)}\n    ilsref=None;truth=None;allwatched=dict(watched)\n    try:\n        for ci,ctx in enumerate(s.contexts(files),1):\n            s.unchanged(files,watched,checks)\n            print(f"[{ci}/33] {ctx[\'scope\']} | rep={ctx[\'rep\']} | n={len(ctx[\'X\'])}",flush=True)\n            if ctx[\'scope\']!=\'LMS_RESAMPLING\':\n                dt,dtest,dr,dsum,dd=direct_job(s,files,out,control,ctx,rules[ctx[\'dataset\']],code,cfg)\n                dcounts[dd]+=1;rows.append(dsum);artifacts.append({\'scope\':ctx[\'scope\'],\'dataset\':ctx[\'dataset\'],\'pathway\':DIRECT,\n                    \'initializer\':\'NONE\',\'rule\':DIRECT,\'rep\':None,\'run\':0,\'files\':json.dumps(list(dr[\'outputs\']),ensure_ascii=False),\'source\':\'individual categorical indicators\'})\n                if ctx[\'scope\']==\'LMS_FULL\':directfull={\'profiles\':dt,\'signature\':dr[\'signature\']}\n                if ctx[\'scope\']==\'PENDIKOM_TRAIN\':\n                    ilsref,truth=s.ils_reference(files,ctx);predictions[DIRECT].append(s.numeric_profiles(dtest))\n            else:\n                row,dispo=direct_stability(s,files,out,control,ctx,directfull,code,cfg);stabilities.append(row);scounts[dispo]+=1\n            for method in s.METHODS:\n                runs=15 if method in s.STOCHASTIC and ctx[\'scope\']!=\'LMS_RESAMPLING\' else 1\n                for run in range(runs):\n                    up=s.upstream_job(files,p25,ctx,method,run);allwatched.update(up[\'watched\'])\n                    primary=primary26(s,files,p26,ctx,up,rules[ctx[\'dataset\']]);allwatched.update(primary[\'watched\'])\n                    for rule in RULES:\n                        pathid=pid(method,rule)\n                        if rule==PRIMARY:\n                            mapping=primary[\'mapping\'];decisions=primary_decisions(s,primary[\'evidence\']);record=primary[\'record\'];counts[\'REUSED_26\']+=1\n                            summary=summarize_mapping(s,ctx,up,mapping,decisions,rule,primary[\'mapping_path\'])\n                        else:\n                            mapping,decisions,record,summary,dispo=cluster_job(s,files,out,control,ctx,up,primary,rule,code,cfg);counts[dispo]+=1\n                        rows.append(summary)\n                        details=decisions.copy()\n                        for name,value in [(\'scope\',ctx[\'scope\']),(\'initializer\',method),(\'run\',run),(\'rep\',ctx[\'rep\']),(\'pathway\',pathid)]:details[name]=value\n                        diagnostics.append(details)\n                        artifacts.append({**up[\'identity\'],\'initializer\':method,\'rule\':rule,\'pathway\':pathid,\n                            \'files\':json.dumps(list(record[\'outputs\']),ensure_ascii=False),\'source\':\'stage 26\' if rule==PRIMARY else \'stage 27; frozen evidence from 26\',\n                            \'source_evidence_file\':primary[\'evidence_path\'],\'source25_files\':json.dumps(list(up[\'record\'][\'outputs\']),ensure_ascii=False)})\n                        if ctx[\'scope\']==\'LMS_FULL\' and run==0:\n                            signature=(s.digest({\'upstream\':up[\'checksum\'],\'mapping\':mapping.to_dict(\'records\')}) if rule==PRIMARY\n                                       else s.digest({\'record\':record[\'signature\'],\'rule\':rule,\'mapping\':mapping.to_dict(\'records\')}))\n                            refs[pathid]={\'profiles\':s.profiles(mapping,up[\'labels\'],ctx[\'ids\'],ctx[\'indices\']),\'signature\':signature}\n                        if ctx[\'scope\']==\'LMS_RESAMPLING\':\n                            if rule==PRIMARY:\n                                row,ww=primary_stability(s,files,p26,ctx,up,primary,refs[pathid]);allwatched.update(ww);dispo=\'REUSED_26\';metrics26=row\n                            else:\n                                row,dispo=alternative_stability(s,files,out,control,ctx,up,mapping,refs[pathid],metrics26,rule,code,cfg)\n                            row={**row,\'pathway\':pathid,\'initializer\':method,\'rule\':rule,\'structurally_invariant_on_resolved\':False}\n                            stabilities.append(row);scounts[dispo]+=1\n                        if ctx[\'scope\']==\'PENDIKOM_TRAIN\':\n                            if rule==PRIMARY:\n                                frame=files.frame(primary[\'find\'](\'04_test_targets.csv\'),True)\n                                if frame.identity.tolist()!=ctx[\'test_ids\']:raise RuntimeError(\'Prediksi primary 26 tidak sejajar dengan ILS test.\')\n                            else:frame=s.profiles(mapping,up[\'test_labels\'],ctx[\'test_ids\'])\n                            predictions[pathid].append(s.numeric_profiles(frame))\n            s.save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'cluster_profile_jobs\':sum(counts.values()),\'mapping_counts\':counts,\n                \'direct_contexts\':sum(dcounts.values()),\'stability_jobs\':len(stabilities),\'last_scope\':ctx[\'scope\']})\n        if sum(counts.values())!=1296 or sum(dcounts.values())!=3 or len(stabilities)!=750:raise RuntimeError(\'Jumlah pekerjaan belum lengkap.\')\n        s.unchanged(files,allwatched,checks)\n        ils,cached=evaluate(s,files,out,control,predictions,ilsref,truth,code,cfg)\n        raw,stability,paired=aggregate_stability(s,stabilities)\n        s.save_csv(out/\'01_ringkasan_pembentukan_profil.csv\',pd.DataFrame(rows));s.save_csv(out/\'02_peta_artefak.csv\',pd.DataFrame(artifacts))\n        evidence=pd.concat(diagnostics,ignore_index=True)\n        s.save_csv(out/\'AUDIT_ARAH/01_keputusan_cluster_dimensi.csv.gz\',evidence)\n        s.save_csv(out/\'AUDIT_ARAH/02_ringkasan_konflik.csv\',pd.DataFrame(rows).loc[lambda x:x.rule.ne(DIRECT)])\n        examples=evidence.loc[evidence.scope.eq(\'LMS_FULL\')&evidence.initializer.eq(\'CARE-KM\')&evidence.run.eq(0)&evidence.cluster_id.eq(2)&evidence.dimension.eq(\'Sensing/Intuitive\')]\n        s.save_csv(out/\'AUDIT_ARAH/03_contoh_C03_LMS.csv\',examples)\n        s.save_json(out/\'AUDIT_ARAH/04_definisi.json\',{\'mode_member_conflict\':\'sign(mode_score) and sign(member_score) oppose; unchanged by reporting operator\',\n            \'reported_member_conflict\':\'selected reported direction opposes nonzero member_score; can change with operator\',\n            \'strong\':\'magnitude 9 or 11; counts and denominator reported separately\',\n            \'unit\':\'cluster x dimension within each partition; repeated/overlapping partitions are not independent learners\',\n            \'direction_consistent_member_first\':\'same existing MEMBER_FIRST rule from notebook 12; not a newly claimed algorithm\'})\n        s.save_csv(out/\'LMS_STABILITAS/01_metrik_per_resample.csv\',raw);s.save_csv(out/\'LMS_STABILITAS/02_ringkasan_25_jalur.csv\',stability)\n        s.save_csv(out/\'LMS_STABILITAS/03_selisih_aturan_deskriptif.csv\',paired)\n        s.save_json(out/\'LMS_STABILITAS/04_protokol.json\',{\'reference\':\'own initializer/operator full run 0\',\n            \'sampling\':\'30 matched 80% overlapping LMS samples, original indices and categorical representation\',\n            \'direct_invariance\':\'directions for the same resolved learner are identical by construction; all-learner agreement still reflects unresolved cases\',\n            \'interpretation\':\'direct structural invariance is not empirical superiority of partition robustness\',\n            \'ARI_NMI_AMI\':\'partition metrics unchanged by semantic operator; reuse stage 26 values; not improvements attributable to profile rules\',\n            \'inference\':\'descriptive paired distributions; no independent-cohort tests on 30 overlapping samples\'})\n        main_ids=[pid(\'CARE-KM\',r) if r!=DIRECT else DIRECT for r in DISPLAY_RULES]\n        main=ils.set_index(\'pathway\').loc[main_ids].reset_index()\n        for metric in s.METRIC_NAMES:main[metric+\'_mean_ci95\']=main.apply(lambda r:f"{100*r[metric+\'_mean\']:.2f} [{100*r[metric+\'_ci95_low\']:.2f}, {100*r[metric+\'_ci95_high\']:.2f}]",axis=1)\n        s.save_csv(out/\'TABEL/01_pembentukan_profil_ILS_utama.csv\',main[[\'pathway\',\'initializer\',\'rule\',\'n_runs\']+[m+\'_mean_ci95\' for m in s.METRIC_NAMES]+[\'coverage_mean\',\'complete_profile_coverage\']])\n        s.save_csv(out/\'TABEL/02_semua_initializer_dan_aturan.csv\',ils)\n        ss=stability.set_index(\'pathway\').loc[main_ids].reset_index();s.save_csv(out/\'TABEL/03_stabilitas_aturan_utama.csv\',ss)\n        figsig=s.digest({\'code\':code,\'ils\':s.sha((out/\'ILS/04_ringkasan_overall.csv\').read_bytes()),\'stability\':s.sha((out/\'LMS_STABILITAS/01_metrik_per_resample.csv\').read_bytes())})\n        if not s.cache_read(files,control/\'gambar.json\',figsig,cfg[\'force\']):s.cache_write(files,control/\'gambar.json\',figsig,figures(s,out,ils,raw),{})\n        s.unchanged(files,allwatched,checks);s.save_json(out/\'PROTOKOL/03_manifest_masukan.json\',allwatched)\n        status={\'status\':\'selesai\',\'cluster_profile_jobs\':1296,\'mapping_counts\':counts,\'direct_contexts\':3,\'direct_counts\':dcounts,\n            \'stability_jobs\':750,\'stability_counts\':scounts,\'n_unique_ils_pathways\':25,\'n_ils_test\':139,\'ils_cached\':cached,\n            \'new_clustering_runs\':0,\'full_protocol\':cfg[\'bootstrap_repetitions\']>=10000,\'output\':str(out),\'finished_utc\':datetime.now(timezone.utc).isoformat()}\n        manifest={str(p.relative_to(root)).replace(\'\\\\\',\'/\'):s.sha(p.read_bytes()) for p in out.rglob(\'*\') if p.is_file() and control not in p.parents}\n        s.save_json(control/\'manifest_keluaran.json\',manifest);s.save_json(control/\'status_eksekusi.json\',status)\n        print(\'Tahap 27 selesai:\',counts,\'| direct:\',dcounts,\'| stabilitas:\',scounts,flush=True)\n        print(\'Hasil:\',out)\n        return {\'ils\':ils,\'main_table\':main,\'stability\':stability,\'status\':status,\'output\':str(out)}\n    except BaseException as error:\n        s.save_json(control/\'status_eksekusi.json\',{\'status\':\'terhenti\',\'cluster_profile_jobs\':sum(counts.values()),\'mapping_counts\':counts,\n            \'error\':type(error).__name__+\': \'+str(error),\'resume\':\'jalankan notebook 27 yang sama; isi cache diperiksa per pekerjaan\'})\n        raise\n'

_profile27 = types.ModuleType('_profile27')
exec(compile(PROFILE27_SOURCE, '_profile27', 'exec'), _profile27.__dict__)


In [5]:
HASIL_27 = _profile27.run_profiles(ROOT_27, KONFIGURASI_27, PROFILE27_SOURCE, _s26)
display(HASIL_27['main_table'][['rule', 'n_runs', 'accuracy_mean_ci95',
                               'balanced_accuracy_mean_ci95', 'macro_f1_mean_ci95',
                               'coverage_mean', 'complete_profile_coverage']])
print(HASIL_27['status'])
print('Folder hasil:', HASIL_27['output'])


[1/33] LMS_FULL | rep=None | n=6658
[2/33] LMS_RESAMPLING | rep=0 | n=5326
[3/33] LMS_RESAMPLING | rep=1 | n=5326
[4/33] LMS_RESAMPLING | rep=2 | n=5326
[5/33] LMS_RESAMPLING | rep=3 | n=5326
[6/33] LMS_RESAMPLING | rep=4 | n=5326
[7/33] LMS_RESAMPLING | rep=5 | n=5326
[8/33] LMS_RESAMPLING | rep=6 | n=5326
[9/33] LMS_RESAMPLING | rep=7 | n=5326
[10/33] LMS_RESAMPLING | rep=8 | n=5326
[11/33] LMS_RESAMPLING | rep=9 | n=5326
[12/33] LMS_RESAMPLING | rep=10 | n=5326
[13/33] LMS_RESAMPLING | rep=11 | n=5326
[14/33] LMS_RESAMPLING | rep=12 | n=5326
[15/33] LMS_RESAMPLING | rep=13 | n=5326
[16/33] LMS_RESAMPLING | rep=14 | n=5326
[17/33] LMS_RESAMPLING | rep=15 | n=5326
[18/33] LMS_RESAMPLING | rep=16 | n=5326
[19/33] LMS_RESAMPLING | rep=17 | n=5326
[20/33] LMS_RESAMPLING | rep=18 | n=5326
[21/33] LMS_RESAMPLING | rep=19 | n=5326
[22/33] LMS_RESAMPLING | rep=20 | n=5326
[23/33] LMS_RESAMPLING | rep=21 | n=5326
[24/33] LMS_RESAMPLING | rep=22 | n=5326
[25/33] LMS_RESAMPLING | rep=23 | n=532

,rule,n_runs,accuracy_mean_ci95,balanced_accuracy_mean_ci95,macro_f1_mean_ci95,coverage_mean,complete_profile_coverage
0,PRIMARY_MODE_FIRST,1,"89.75 [87.05, 92.27]","81.29 [76.52, 85.76]","82.98 [77.61, 87.28]",1.000000,1.000000
1,DIRECT_BEHAVIORAL_RULES,1,"73.56 [69.78, 77.34]","71.76 [66.81, 76.55]","73.88 [68.73, 78.30]",0.881295,0.604317
2,MEMBER_ONLY,1,"89.57 [86.87, 92.09]","80.56 [75.90, 84.97]","82.41 [77.04, 86.85]",0.996403,0.985612
3,DIRECTION_CONSISTENT_MEMBER_FIRST,1,"89.75 [87.05, 92.27]","81.29 [76.52, 85.76]","82.98 [77.61, 87.28]",1.000000,1.000000


{'status': 'selesai', 'cluster_profile_jobs': 1296, 'mapping_counts': {'REUSED_26': 432, 'MAP': 864, 'CACHE': 0}, 'direct_contexts': 3, 'direct_counts': {'DIRECT': 3, 'CACHE': 0}, 'stability_jobs': 750, 'stability_counts': {'REUSED_26': 240, 'CALCULATE': 510, 'CACHE': 0}, 'n_unique_ils_pathways': 25, 'n_ils_test': 139, 'ils_cached': False, 'new_clustering_runs': 0, 'full_protocol': True, 'output': '/content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/27_Perbandingan_Pembentukan_Profil', 'finished_utc': '2026-11-06T12:23:22.399317+00:00'}
Folder hasil: /content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/27_Perbandingan_Pembentukan_Profil


## Membaca hasil

- `TABEL/01_pembentukan_profil_ILS_utama.csv`: empat cara pembentukan profil,
  dengan CARE-KM tetap untuk tiga operator berbasis cluster.
- `TABEL/02_semua_initializer_dan_aturan.csv`: 25 jalur unik pada ILS.
- `TABEL/03_stabilitas_aturan_utama.csv`: ringkasan empat jalur utama pada LMS.
- `ILS/`: prediksi per peserta, metrik per run/dimensi, confusion matrices,
  reference counts, paired differences, bank metrik dan bobot bootstrap.
- `AUDIT_ARAH/`: keputusan cluster-dimensi, konflik, dan contoh C03 LMS.
- `LMS_STABILITAS/`: 750 hasil per resample, ringkasan 25 jalur, dan paired
  differences deskriptif. Artefak primary tetap merujuk sumber tahap 26.
- `BANK/`: mapping alternatif dan target train/test; target dikompresi sebagai
  `.csv.gz`. Mapping primary merujuk bank 26 pada peta artefak.
- `DIRECT/`: profil dan evidence indikator peserta untuk kontrol tanpa cluster.
- `01_ringkasan_pembentukan_profil.csv`, `02_peta_artefak.csv`: hubungan sumber,
  scope, initializer, operator, run/rep, dan berkas hasil.
- `GAMBAR/`, `PROTOKOL/`, `KONTROL/`: PNG/PDF, konfigurasi, manifest, dan cache.

Jika Colab terputus, jalankan notebook yang sama kembali. Isi sumber, kode,
parameter, versi, serta artefak diperiksa sebelum cache dipakai. Perubahan
tampilan output notebook tidak membatalkan cache. Artefak sumber 25/26 yang
berubah ditolak dan perlu dipulihkan atau dihasilkan kembali pada tahapnya.
